# HoloSyn Visual Interface (Gradio) — Integrates Your Distilled TorchScript Model

This Colab notebook builds a **visual UI** to:
- Load your distilled model: `/mnt/data/student_distilled_heads_hf.torchscript.pt`
- Load normalization: `/mnt/data/student_norm_hf.json`
- Optionally load your archive: `/mnt/data/Archive.zip` and browse files
- Compute modality-specific features (text/audio/image/video/haptics)
- Run inference → **valence/arousal/calm/trust**
- Visualize:
  - meters + time series
  - optional **two-peer synchrony** (A/B)
- Export a JSON session log

**Privacy note:** Everything runs locally in the notebook runtime.

---

## Inputs expected
- `/mnt/data/student_distilled_heads_hf.torchscript.pt`
- `/mnt/data/student_norm_hf.json`
- Optional: `/mnt/data/Archive.zip`


In [ ]:
#@title 0) Install deps
#@title 0) Install deps
!pip -q install -U gradio numpy "pandas==2.2.2" pillow opencv-python soundfile librosa ffmpeg-python sentence-transformers transformers
!pip -q install -U torch torchvision torchaudio pyarrow cirq
import os, json, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import librosa
import soundfile as sf
import cv2

import gradio as gr
print("✅ Installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 107.1 MB/s eta 0:00:00
✅ Installed


In [ ]:
#@title 1) Paths + load model
MODEL_PATH = "/content/student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "/content/student_norm_hf.json"
ARCHIVE_ZIP = "/content/Archive.zip"

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)

print("✅ Model loaded")
print("Feature dims:", len(NUMERIC_COLS))

✅ Model loaded
Feature dims: 789


In [ ]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(k, len(INDEX[k]))


In [ ]:
#@title 3) Feature extraction (must align with training feature schema)
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    # Return dict of features; missing features are 0.0.
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if isinstance(path_or_text, str) and "\n" in path_or_text else load_text(path_or_text)
        feats.update(featurize_text(s))
        preview = s[:1000]
    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]
    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"
    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")
    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None
    else:
        preview = f"Unsupported modality for: {path_or_text}"

    # Build full vector in the exact order expected by the student
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
        else:
            # embeddings columns (clip_*, w2v_*) are not computed here; keep 0 unless you add embed models
            x[i] = 0.0
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    # [valence, arousal, calm, trust]
    return y

In [ ]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()}
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "/mnt/data/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out

## 5) Gradio UI

Two panels:
- **Single input inference** (pick modality + source)
- **Two-peer synchrony**: run A & B and compute cosine similarity on `[valence, arousal, calm, trust]`


In [ ]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        # preview might be PIL Image
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def infer_pair(modalityA, fileA, textA, modalityB, fileB, textB):
    prevA, vA, aA, cA, tA, featsA = infer_one(modalityA, fileA, textA)
    prevB, vB, aB, cB, tB, featsB = infer_one(modalityB, fileB, textB)
    eA = np.array([vA,aA,cA,tA], dtype=np.float32)
    eB = np.array([vB,aB,cB,tB], dtype=np.float32)
    sync = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))
    return prevA, prevB, sync

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")
        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame) OR Text snippet", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)
        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.Dropdown.update(choices=list_options(m), value=None)
        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        def render(preview_obj, v,a,c,t, feats):
            # If preview is an image, show it; else show text in preview_txt
            if isinstance(preview_obj, Image.Image):
                return preview_obj, "", v,a,c,t, feats
            else:
                # show placeholder image blank
                return None, str(preview_obj), v,a,c,t, feats

        run_btn.click(
            fn=lambda m,f,txt: infer_one(m,f,txt),
            inputs=[modality,file_dd,free_text],
            outputs=[preview_txt,val,aro,calm,trust,feats_json],
        ).then(
            fn=lambda prev_txt, v,a,c,t, feats: render(prev_txt, v,a,c,t, feats),
            inputs=[preview_txt,val,aro,calm,trust,feats_json],
            outputs=[preview,preview_txt,val,aro,calm,trust,feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")
        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")
        run_pair = gr.Button("Run pair + synchrony")
        with gr.Row():
            prevA = gr.Image(label="Preview A", type="pil")
            prevB = gr.Image(label="Preview B", type="pil")
        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        run_pair.click(
            fn=infer_pair,
            inputs=[modalityA,fileA,textA, modalityB,fileB,textB],
            outputs=[prevA,prevB,sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

In [ ]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.Dropdown.update(choices=list_options(m), value=None)
        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        # --- FIX: Unified handler to safely route text vs images ---
        def process_single(m, f, txt):
            prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
            if isinstance(prev_obj, Image.Image):
                return prev_obj, "", v, a, c, t, feats
            else:
                return None, str(prev_obj), v, a, c, t, feats

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview, preview_txt, val, aro, calm, trust, feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")

        run_pair = gr.Button("Run pair + synchrony")

        # --- FIX: Provide both Image and Text preview blocks for safely rendering dynamic modalities ---
        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image/Video)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text/Other)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image/Video)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text/Other)", lines=4)

        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        # --- FIX: Unified handler to safely route pairs ---
        def process_pair(mA, fA, txtA, mB, fB, txtB):
            prevA, vA, aA, cA, tA, featsA = infer_one(mA, fA, txtA)
            prevB, vB, aB, cB, tB, featsB = infer_one(mB, fB, txtB)

            eA = np.array([vA,aA,cA,tA], dtype=np.float32)
            eB = np.array([vB,aB,cB,tB], dtype=np.float32)
            sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))

            outA_img = prevA if isinstance(prevA, Image.Image) else None
            outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
            outB_img = prevB if isinstance(prevB, Image.Image) else None
            outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

            return outA_img, outA_txt, outB_img, outB_txt, sync_val

        run_pair.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

In [ ]:
#@title 1) Imports & Load Model
import os, json, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import librosa
import soundfile as sf
import cv2
import gradio as gr
from sentence_transformers import SentenceTransformer

MODEL_PATH = "./student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "./student_norm_hf.json"
ARCHIVE_ZIP = "./Archive.zip"

print("Loading text embedding model (this may take a moment)...")
embedder = SentenceTransformer('all-mpnet-base-v2')

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)

print("✅ Model and Norm loaded")
print("Feature dims:", len(NUMERIC_COLS))

KeyboardInterrupt: 

In [ ]:
#@title 1) Imports & Load Model
import os, json, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import librosa
import soundfile as sf
import cv2
import gradio as gr
from sentence_transformers import SentenceTransformer
import transformers
import huggingface_hub

print(f"Transformers version: {transformers.__version__}")
print(f"Hugging Face Hub version: {huggingface_hub.__version__}")

MODEL_PATH = "./student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "./student_norm_hf.json"
ARCHIVE_ZIP = "./Archive.zip"

print("Loading text embedding model (this may take a moment)...")
embedder = SentenceTransformer('all-mpnet-base-v2')

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)

print("✅ Model and Norm loaded")
print("Feature dims:", len(NUMERIC_COLS))

Transformers version: 5.4.0
Hugging Face Hub version: 1.7.1
Loading text embedding model (this may take a moment)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model and Norm loaded
Feature dims: 789


In [ ]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(k, len(INDEX[k]))

Archive present: True
audio 49
video 54
image 133
text 49
haptics 98
other 25


In [ ]:
#@title 3) Feature extraction
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if isinstance(path_or_text, str) and "\n" in path_or_text else load_text(path_or_text)
        feats.update(featurize_text(s))
        # Compute the 768-d embedding
        emb = embedder.encode(s)
        for i in range(768):
            feats[f"w2v_{i}"] = float(emb[i])
        preview = s[:1000]

    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]
    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"
    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")
    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None
    else:
        preview = f"Unsupported modality for: {path_or_text}"

    # Build full vector
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
        else:
            x[i] = 0.0
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    return y

In [ ]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()}
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "/mnt/data/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out

In [ ]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.update(choices=list_options(m), value=None)

        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        def process_single(m, f, txt):
            prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
            if isinstance(prev_obj, Image.Image):
                return prev_obj, "", v, a, c, t, feats
            else:
                return None, str(prev_obj), v, a, c, t, feats

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview, preview_txt, val, aro, calm, trust, feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")

        run_pair = gr.Button("Run pair + synchrony")

        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image/Video)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text/Other)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image/Video)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text/Other)", lines=4)

        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        def process_pair(mA, fA, txtA, mB, fB, txtB):
            prevA, vA, aA, cA, tA, featsA = infer_one(mA, fA, txtA)
            prevB, vB, aB, cB, tB, featsB = infer_one(mB, fB, txtB)

            eA = np.array([vA,aA,cA,tA], dtype=np.float32)
            eB = np.array([vB,aB,cB,tB], dtype=np.float32)
            sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))

            outA_img = prevA if isinstance(prevA, Image.Image) else None
            outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
            outB_img = prevB if isinstance(prevB, Image.Image) else None
            outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

            return outA_img, outA_txt, outB_img, outB_txt, sync_val

        run_pair.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d759fb73038577b0d3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(k, len(INDEX[k]))

In [ ]:
#@title 3) Feature extraction
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if isinstance(path_or_text, str) and "\n" in path_or_text else load_text(path_or_text)
        feats.update(featurize_text(s))
        # Compute the 768-d embedding
        emb = embedder.encode(s)
        for i in range(768):
            feats[f"w2v_{i}"] = float(emb[i])
        preview = s[:1000]

    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]
    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"
    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")
    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None
    else:
        preview = f"Unsupported modality for: {path_or_text}"

    # Build full vector
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
        else:
            x[i] = 0.0
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    return y

In [ ]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()}
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "/mnt/data/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out

In [ ]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.update(choices=list_options(m), value=None)

        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        def process_single(m, f, txt):
            prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
            if isinstance(prev_obj, Image.Image):
                return prev_obj, "", v, a, c, t, feats
            else:
                return None, str(prev_obj), v, a, c, t, feats

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview, preview_txt, val, aro, calm, trust, feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")

        run_pair = gr.Button("Run pair + synchrony")

        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image/Video)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text/Other)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image/Video)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text/Other)", lines=4)

        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        def process_pair(mA, fA, txtA, mB, fB, txtB):
            prevA, vA, aA, cA, tA, featsA = infer_one(mA, fA, txtA)
            prevB, vB, aB, cB, tB, featsB = infer_one(mB, fB, txtB)

            eA = np.array([vA,aA,cA,tA], dtype=np.float32)
            eB = np.array([vB,aB,cB,tB], dtype=np.float32)
            sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))

            outA_img = prevA if isinstance(prevA, Image.Image) else None
            outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
            outB_img = prevB if isinstance(prevB, Image.Image) else None
            outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

            return outA_img, outA_txt, outB_img, outB_txt, sync_val

        run_pair.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a800f6c0cef6f4e514.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install cirq qsimcirq torch torchvision numpy
import torch
import torch.nn as nn
import torch.nn.functional as F
import cirq
import qsimcirq
import json
import numpy as np

In [ ]:
# Load your multimodal normalization statistics
with open('student_norm_hf.json', 'r') as f:
    norm_stats = json.load(f)

# Extract means and standard deviations for PyTorch preprocessing
mu = torch.tensor(norm_stats['mu'], dtype=torch.float32)
sd = torch.tensor(norm_stats['sd'], dtype=torch.float32)

def normalize_features(features):
    # Avoid division by zero for any zero-variance features
    return (features - mu) / (sd + 1e-7)

In [ ]:
class QSimLayer(nn.Module):
    def __init__(self, num_qubits):
        super(QSimLayer, self).__init__()
        self.num_qubits = num_qubits
        self.qubits = cirq.GridQubit.rect(1, num_qubits)
        self.simulator = qsimcirq.QSimSimulator()

        # Trainable PyTorch parameters for the quantum circuit
        self.theta = nn.Parameter(torch.rand(num_qubits) * np.pi)

    def forward(self, x):
        batch_size = x.size(0)
        out = torch.zeros(batch_size, self.num_qubits, device=x.device)

        # Iterating through the batch to simulate the quantum state
        for i in range(batch_size):
            circuit = cirq.Circuit()

            # 1. Feature Encoding: Mapping continuous variables (e.g., w2v) to Rx gates
            for j in range(self.num_qubits):
                feature_val = x[i, j].item()
                circuit.append(cirq.rx(feature_val)(self.qubits[j]))

            # 2. Trainable Variational Entanglement Layer
            for j in range(self.num_qubits - 1):
                circuit.append(cirq.CNOT(self.qubits[j], self.qubits[j+1]))
            for j in range(self.num_qubits):
                circuit.append(cirq.ry(self.theta[j].item())(self.qubits[j]))

            # 3. Simulate with qsimcirq for state-vector processing
            result = self.simulator.simulate(circuit)
            state_vector = result.state_vector()

            # Extract probabilities and project back to a real tensor
            probs = np.abs(state_vector)**2
            out[i] = torch.tensor(probs[:self.num_qubits], dtype=torch.float32)

        return out

In [ ]:
# 1. Load the Teacher Model (e.g., your open source model or holosyn_heads)
teacher_model = torch.jit.load('holosyn_heads.torchscript.pt')
teacher_model.eval() # Freeze the teacher

# 2. Load the Student Model
student_model = torch.jit.load('student_distilled_heads_hf.torchscript.pt')
student_model.train()

# 3. Wrap student with the Quantum Layer
class QuantumDistilledStudent(nn.Module):
    def __init__(self, base_student, num_q_features=8):
        super(QuantumDistilledStudent, self).__init__()
        self.num_q_features = num_q_features
        # Routing a subset of features (like w2v embeddings) through the quantum layer
        self.quantum_layer = QSimLayer(num_qubits=num_q_features)
        self.base_student = base_student

    def forward(self, x):
        # Pass the first 'n' features through the VQC
        q_out = self.quantum_layer(x[:, :self.num_q_features])

        # Re-inject quantum features into the main tensor
        x_modified = x.clone()
        x_modified[:, :self.num_q_features] = q_out

        return self.base_student(x_modified)

model = QuantumDistilledStudent(student_model)

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, T=2.0, alpha=0.5):
    """
    Combines KL Divergence from the teacher and Standard Loss from the labels.
    """
    hard_loss = F.cross_entropy(student_logits, labels)
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction='batchmean'
    ) * (T * T)

    return alpha * hard_loss + (1. - alpha) * soft_loss

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Pseudo-training execution
# for batch in dataloader:
#     features, labels = batch
#     features = normalize_features(features)
#
#     with torch.no_grad():
#         teacher_preds = teacher_model(features)
#
#     student_preds = model(features)
#     loss = distillation_loss(student_preds, teacher_preds, labels)
#
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

In [ ]:
class QuantumAutogradFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, inputs, weights, simulator, qubits):
        """
        inputs: Input features mapped to the circuit.
        weights: Trainable parameters (theta).
        simulator: The qsimcirq instance.
        qubits: The grid of Cirq qubits.
        """
        ctx.save_for_backward(inputs, weights)
        ctx.simulator = simulator
        ctx.qubits = qubits

        batch_size = inputs.size(0)
        num_qubits = len(qubits)
        out = torch.zeros(batch_size, num_qubits, device=inputs.device)

        # Forward pass execution
        for i in range(batch_size):
            circuit = cirq.Circuit()

            # Encode inputs
            for j in range(num_qubits):
                circuit.append(cirq.rx(inputs[i, j].item())(qubits[j]))

            # Apply weights (trainable Entanglement Layer)
            for j in range(num_qubits - 1):
                circuit.append(cirq.CNOT(qubits[j], qubits[j+1]))
            for j in range(num_qubits):
                circuit.append(cirq.ry(weights[j].item())(qubits[j]))

            result = simulator.simulate(circuit)
            probs = np.abs(result.state_vector())**2
            out[i] = torch.tensor(probs[:num_qubits], dtype=torch.float32)

        return out

    @staticmethod
    def backward(ctx, grad_output):
        """
        Calculates gradients using the Parameter-Shift Rule.
        """
        inputs, weights = ctx.saved_tensors
        simulator = ctx.simulator
        qubits = ctx.qubits

        batch_size = inputs.size(0)
        num_qubits = len(qubits)

        # We only need gradients for the trainable weights, not the simulator objects
        grad_weights = torch.zeros_like(weights)

        # Shift magnitude
        shift = np.pi / 2.0

        # Calculate gradients for each parameter
        for p_idx in range(len(weights)):
            # Create shifted parameter tensors
            weight_plus = weights.clone()
            weight_minus = weights.clone()

            weight_plus[p_idx] += shift
            weight_minus[p_idx] -= shift

            # We must execute the batch for both the + shift and - shift
            out_plus = torch.zeros(batch_size, num_qubits, device=inputs.device)
            out_minus = torch.zeros(batch_size, num_qubits, device=inputs.device)

            for i in range(batch_size):
                # Build and simulate circuit for + shift
                circ_plus = cirq.Circuit()
                for j in range(num_qubits): circ_plus.append(cirq.rx(inputs[i, j].item())(qubits[j]))
                for j in range(num_qubits - 1): circ_plus.append(cirq.CNOT(qubits[j], qubits[j+1]))
                for j in range(num_qubits): circ_plus.append(cirq.ry(weight_plus[j].item())(qubits[j]))

                res_plus = simulator.simulate(circ_plus)
                out_plus[i] = torch.tensor(np.abs(res_plus.state_vector())**2[:num_qubits])

                # Build and simulate circuit for - shift
                circ_minus = cirq.Circuit()
                for j in range(num_qubits): circ_minus.append(cirq.rx(inputs[i, j].item())(qubits[j]))
                for j in range(num_qubits - 1): circ_minus.append(cirq.CNOT(qubits[j], qubits[j+1]))
                for j in range(num_qubits): circ_minus.append(cirq.ry(weight_minus[j].item())(qubits[j]))

                res_minus = simulator.simulate(circ_minus)
                out_minus[i] = torch.tensor(np.abs(res_minus.state_vector())**2[:num_qubits])

            # Apply the parameter shift rule formula
            gradients = 0.5 * (out_plus - out_minus)

            # Chain rule: multiply the quantum gradients by the incoming gradients from the student model
            grad_weights[p_idx] = torch.sum(gradients * grad_output)

        # Return gradients for each input to `forward`. None for inputs that don't require gradients.
        # Gradients for: (inputs, weights, simulator, qubits)
        return None, grad_weights, None, None

<>:77: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
<>:86: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
<>:77: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
<>:86: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
/tmp/ipykernel_19021/3678387536.py:77: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
  out_plus[i] = torch.tensor(np.abs(res_plus.state_vector())**2[:num_qubits])
/tmp/ipykernel_19021/3678387536.py:86: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
  out_minus[i] = torch.tensor(np.abs(res_minus.state_vector())**2[:num_qubits])


In [ ]:
class DifferentiableQSimLayer(nn.Module):
    def __init__(self, num_qubits):
        super(DifferentiableQSimLayer, self).__init__()
        self.num_qubits = num_qubits
        self.qubits = cirq.GridQubit.rect(1, num_qubits)
        self.simulator = qsimcirq.QSimSimulator()

        # Initialize trainable parameters
        self.theta = nn.Parameter(torch.rand(num_qubits) * np.pi)

    def forward(self, x):
        # Call the custom Autograd function
        return QuantumAutogradFunction.apply(x, self.theta, self.simulator, self.qubits)

In [ ]:
!pip install brian2 requests

import torch
from brian2 import *
import requests
import time
import numpy as np

# Set Brian2 to use numpy for standard execution, or cython for performance later
prefs.codegen.target = 'numpy'

In [ ]:
def fetch_starlink_telemetry():
    """
    Fetches real-time telemetry from the local Starlink Dish API.
    Returns a normalized tensor of features: [ping_drop_rate, ping_latency, downlink_bps, uplink_bps]
    """
    try:
        # Standard local IP for Starlink Dishy
        response = requests.get("http://192.168.100.1/api/v1/status", timeout=2)
        data = response.json()

        # Extract raw features (mocking the structure if offline)
        drop_rate = data.get('popPingDropRate', 0.0)
        latency = data.get('popPingLatencyMs', 40.0)
        downlink = data.get('downlinkThroughputBps', 1000000)
        uplink = data.get('uplinkThroughputBps', 500000)

    except requests.exceptions.RequestException:
        # Fallback simulated data for testing the pipeline when offline
        drop_rate = np.random.uniform(0.0, 0.05)
        latency = np.random.uniform(20.0, 80.0)
        downlink = np.random.uniform(1e6, 50e6)
        uplink = np.random.uniform(5e5, 10e6)

    # Heuristic normalization to map into a reasonable range for quantum rotation gates [-pi, pi]
    features = [
        drop_rate * 10,                 # Scale drop rate
        (latency - 40) / 20,            # Normalize latency around 40ms
        np.log10(downlink + 1) / 8,     # Log scale for bps
        np.log10(uplink + 1) / 7
    ]

    # Return as a batch of size 1
    return torch.tensor([features], dtype=torch.float32)

In [ ]:
def setup_brian2_snn(num_input_neurons, num_hidden_neurons=20):
    start_scope() # Clear the Brian2 registry for a fresh run

    # 1. The LIF Neuron Equation
    tau = 10*ms
    eqs = '''
    dv/dt = (1 - v) / tau : 1 (unless refractory)
    '''

    # 2. Network Layers
    # Input layer driven by Poisson rates derived from the Quantum Layer
    P_input = PoissonGroup(num_input_neurons, rates=0*Hz)

    # Hidden processing layer
    G_hidden = NeuronGroup(num_hidden_neurons, eqs, threshold='v>0.8', reset='v = 0', refractory=2*ms, method='exact')

    # 3. Synapses (Connections)
    S = Synapses(P_input, G_hidden, 'w : 1', on_pre='v_post += w')
    S.connect(p=0.5) # 50% sparsity
    S.w = 'rand() * 0.2' # Random initial weights

    # 4. Monitors for tracking the engine's output
    spike_monitor = SpikeMonitor(G_hidden)
    state_monitor = StateMonitor(G_hidden, 'v', record=True)

    network = Network(P_input, G_hidden, S, spike_monitor, state_monitor)
    return network, P_input, spike_monitor

In [ ]:
def setup_brian2_snn(num_input_neurons, num_hidden_neurons=20):
    start_scope() # Clear the Brian2 registry for a fresh run

    # 1. The LIF Neuron Equation
    tau = 10*ms
    eqs = '''
    dv/dt = (1 - v) / tau : 1 (unless refractory)
    '''

    # 2. Network Layers
    P_input = PoissonGroup(num_input_neurons, rates=0*Hz)

    # Explicitly pass the namespace here to fix the KeyError
    G_hidden = NeuronGroup(num_hidden_neurons, eqs, threshold='v>0.8', reset='v = 0',
                           refractory=2*ms, method='exact', namespace={'tau': tau})

    # 3. Synapses (Connections)
    S = Synapses(P_input, G_hidden, 'w : 1', on_pre='v_post += w')
    S.connect(p=0.5)
    S.w = 'rand() * 0.2'

    # 4. Monitors
    spike_monitor = SpikeMonitor(G_hidden)
    state_monitor = StateMonitor(G_hidden, 'v', record=True)

    network = Network(P_input, G_hidden, S, spike_monitor, state_monitor)
    return network, P_input, spike_monitor

In [ ]:
class SpikingSurrogateAutograd(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input_rates, synaptic_weights, snn_network, S_synapse, monitor):
        """
        Forward pass: Sync PyTorch weights to Brian2, run simulation, return spike counts.
        """
        # 1. Sync PyTorch weights -> Brian2 SNN
        # S_synapse.w expects a flat numpy array
        S_synapse.w = synaptic_weights.detach().cpu().numpy()

        # 2. Set input rates and run
        # Assuming input_rates is mapped to a PoissonGroup (snn_network.objects[0])
        poisson_group = next(obj for obj in snn_network.objects if isinstance(obj, PoissonGroup))
        poisson_group.rates = input_rates.detach().cpu().numpy() * Hz

        # Reset and run the network
        snn_network.restore('initial_state') # Ensure we start fresh each batch
        snn_network.run(100*ms)

        # 3. Extract outputs (spike counts per neuron)
        spike_counts = torch.tensor(monitor.count[:], dtype=torch.float32, device=input_rates.device)

        # Save context for the backward pass
        ctx.save_for_backward(input_rates, synaptic_weights, spike_counts)
        ctx.S_synapse = S_synapse

        return spike_counts

    @staticmethod
    def backward(ctx, grad_output):
        """
        Backward pass: Approximate the gradients for the synaptic weights.
        """
        input_rates, synaptic_weights, spike_counts = ctx.saved_tensors
        S_synapse = ctx.S_synapse

        # Surrogate Gradient Approximation
        # Since exact Backprop-Through-Time (BPTT) is complex in Brian2,
        # we approximate the gradient of the weights based on the pre/post firing rates.
        # Simple Hebbian-like pseudo-gradient: grad_w ~ grad_out * pre_activity

        # Extract pre-synaptic and post-synaptic indices
        sources = S_synapse.i[:]
        targets = S_synapse.j[:]

        grad_weights = torch.zeros_like(synaptic_weights)

        # Accumulate gradients for each synapse based on the error at the target neuron
        # and the activity of the source neuron (input rate).
        for idx in range(len(sources)):
            pre_neuron = sources[idx]
            post_neuron = targets[idx]

            # The surrogate derivative: error * input_activity
            # Adding a small constant to ensure non-zero flow
            surrogate_derivative = grad_output[post_neuron] * (input_rates[pre_neuron] + 1e-4)
            grad_weights[idx] = surrogate_derivative

        return None, grad_weights, None, None, None

In [ ]:
class Brian2PyTorchWrapper(nn.Module):
    def __init__(self, snn_network, P_input, S_synapse, monitor):
        super(Brian2PyTorchWrapper, self).__init__()
        self.snn_network = snn_network
        self.P_input = P_input
        self.S_synapse = S_synapse
        self.monitor = monitor

        # Store the initial state of the network so we can reset it every forward pass
        self.snn_network.store('initial_state')

        # Register the Brian2 weights as a trainable PyTorch parameter
        initial_weights = torch.tensor(S_synapse.w[:], dtype=torch.float32)
        self.synaptic_weights = nn.Parameter(initial_weights)

    def forward(self, x):
        return SpikingSurrogateAutograd.apply(
            x,
            self.synaptic_weights,
            self.snn_network,
            self.S_synapse,  # Fixed the double 'self' typo here
            self.monitor
        )

In [ ]:
def setup_brian2_snn(num_input_neurons, num_hidden_neurons=20):
    start_scope() # Clear the Brian2 registry for a fresh run

    # 1. The LIF Neuron Equation
    # We bake the 10*ms time constant directly into the string to avoid ALL namespace scope issues
    eqs = '''
    dv/dt = (1 - v) / (10*ms) : 1 (unless refractory)
    '''

    # 2. Network Layers
    P_input = PoissonGroup(num_input_neurons, rates=0*Hz)

    G_hidden = NeuronGroup(num_hidden_neurons, eqs, threshold='v>0.8', reset='v = 0', refractory=2*ms, method='exact')

    # 3. Synapses (Connections)
    S = Synapses(P_input, G_hidden, 'w : 1', on_pre='v_post += w')
    S.connect(p=0.5)
    S.w = 'rand() * 0.2'

    # 4. Monitors
    spike_monitor = SpikeMonitor(G_hidden)
    state_monitor = StateMonitor(G_hidden, 'v', record=True)

    network = Network(P_input, G_hidden, S, spike_monitor, state_monitor)
    return network, P_input, spike_monitor

In [ ]:
num_telemetry_features = 4
snn_network, snn_input, snn_monitor = setup_brian2_snn(num_input_neurons=num_telemetry_features)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import cirq
import qsimcirq
from brian2 import *
import numpy as np
import time

# ---------------------------------------------------------
# 1. SNN Setup (Namespace and Set-Indexing Fixed)
# ---------------------------------------------------------
prefs.codegen.target = 'numpy'

def setup_brian2_snn(num_input_neurons, num_hidden_neurons=20):
    start_scope()

    # Time constant is hardcoded into the string to avoid namespace KeyError
    eqs = '''
    dv/dt = (1 - v) / (10*ms) : 1 (unless refractory)
    '''

    P_input = PoissonGroup(num_input_neurons, rates=0*Hz)
    G_hidden = NeuronGroup(num_hidden_neurons, eqs, threshold='v>0.8', reset='v = 0', refractory=2*ms, method='exact')

    S = Synapses(P_input, G_hidden, 'w : 1', on_pre='v_post += w')
    S.connect(p=0.5)
    S.w = 'rand() * 0.2'

    spike_monitor = SpikeMonitor(G_hidden)
    state_monitor = StateMonitor(G_hidden, 'v', record=True)

    network = Network(P_input, G_hidden, S, spike_monitor, state_monitor)
    return network, P_input, spike_monitor

# ---------------------------------------------------------
# 2. PyTorch to Brian2 Autograd Bridge
# ---------------------------------------------------------
class SpikingSurrogateAutograd(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input_rates, synaptic_weights, snn_network, S_synapse, monitor):
        S_synapse.w = synaptic_weights.detach().cpu().numpy()

        poisson_group = next(obj for obj in snn_network.objects if isinstance(obj, PoissonGroup))
        poisson_group.rates = input_rates.detach().cpu().numpy() * Hz

        snn_network.restore('initial_state')
        snn_network.run(100*ms)

        spike_counts = torch.tensor(monitor.count[:], dtype=torch.float32, device=input_rates.device)

        ctx.save_for_backward(input_rates, synaptic_weights, spike_counts)
        ctx.S_synapse = S_synapse
        return spike_counts

    @staticmethod
    def backward(ctx, grad_output):
        input_rates, synaptic_weights, _ = ctx.saved_tensors
        S_synapse = ctx.S_synapse

        sources = S_synapse.i[:]
        targets = S_synapse.j[:]

        grad_weights = torch.zeros_like(synaptic_weights)
        for idx in range(len(sources)):
            pre_neuron = sources[idx]
            post_neuron = targets[idx]
            # Surrogate derivative calculation
            grad_weights[idx] = grad_output[post_neuron] * (input_rates[pre_neuron] + 1e-4)

        return None, grad_weights, None, None, None

class Brian2PyTorchWrapper(nn.Module):
    def __init__(self, snn_network, P_input, S_synapse, monitor):
        super(Brian2PyTorchWrapper, self).__init__()
        self.snn_network = snn_network
        self.P_input = P_input
        self.S_synapse = S_synapse
        self.monitor = monitor

        self.snn_network.store('initial_state')

        initial_weights = torch.tensor(S_synapse.w[:], dtype=torch.float32)
        self.synaptic_weights = nn.Parameter(initial_weights)

    def forward(self, x):
        return SpikingSurrogateAutograd.apply(
            x, self.synaptic_weights, self.snn_network, self.S_synapse, self.monitor
        )

# ---------------------------------------------------------
# 3. Execution & Training Pipeline
# ---------------------------------------------------------
# Assume q_layer is your DifferentiableQSimLayer defined earlier
# Assume student_model and teacher_model are loaded
# Assume fetch_starlink_telemetry() and distillation_loss() are defined

num_telemetry_features = 4
snn_net, snn_inp, snn_mon = setup_brian2_snn(num_input_neurons=num_telemetry_features)

# Dynamically find the Synapses object to avoid the 'set' TypeError
snn_synapse_obj = next(obj for obj in snn_net.objects if isinstance(obj, Synapses))
snn_module = Brian2PyTorchWrapper(snn_net, snn_inp, snn_synapse_obj, snn_mon)

# Add your optimizer and training loop here...

In [ ]:
!pip install gradio sentence-transformers

import gradio as gr
import torch

# For generating mock 768-dimensional embeddings to represent your LLM/W2V text input
# In production, replace this with your actual Gemini API call or local embedding model
class LLMEmbeddingMock:
    def encode(self, text):
        # Returns a normalized 768-dimensional tensor
        return torch.rand(1, 768)

llm_embedder = LLMEmbeddingMock()

def process_multimodal_forward_pass(text_input, snn_module, q_layer, student_model):
    """
    Executes the full forward pass: Telemetry -> Quantum -> SNN + LLM Text -> Student
    """
    # 1. Fetch & Quantum Encode Telemetry
    telemetry_tensor = fetch_starlink_telemetry()
    with torch.no_grad():
        q_probs = q_layer(telemetry_tensor)
        q_rates = q_probs.squeeze() * 100.0

    # 2. SNN Forward Pass (Output shape: [1, num_hidden_neurons])
    spike_counts = snn_module(q_rates).unsqueeze(0)

    # 3. LLM Text Embedding (Output shape: [1, 768])
    text_embeddings = llm_embedder.encode(text_input)

    # 4. The Concatenation Bridge
    # Merges the spatial SNN spikes with the dense LLM embeddings.
    # Resulting shape: [1, num_hidden_neurons + 768]
    combined_features = torch.cat([spike_counts, text_embeddings], dim=1)

    # Note: Ensure your student_model's input layer is sized to accept
    # (num_hidden_neurons + 768) features to match this concatenated tensor!

    # 5. Final Student Prediction
    with torch.no_grad():
        student_logits = student_model(combined_features)

    return spike_counts, student_logits

In [ ]:
def engine_ui_inference(uploaded_file, text_prompt):
    """
    The function triggered when you click 'Run Engine' in Gradio.
    """
    # Handle optional file upload
    file_content = ""
    if uploaded_file is not None:
        try:
            with open(uploaded_file.name, 'r') as f:
                file_content = f.read()
        except Exception as e:
            file_content = f"Error reading file: {e}"

    # Combine uploaded text with the manual prompt
    full_text_input = f"{text_prompt}\n\nContext from file: {file_content[:500]}..."

    # Run the Wanalytics pipeline
    try:
        spike_counts, logits = process_multimodal_forward_pass(
            full_text_input, snn_module, q_layer, student_model
        )

        # Format the output for the UI
        spikes_out = int(spike_counts.sum().item())
        predicted_class = torch.argmax(logits, dim=1).item()

        results = (
            f"🟢 Pipeline Execution Successful\n"
            f"-----------------------------------\n"
            f"Starlink -> Quantum SNN Spikes Fired: {spikes_out}\n"
            f"Student Model Raw Logits: {logits.numpy().tolist()}\n"
            f"Predicted Output Class: {predicted_class}"
        )
        return results

    except Exception as e:
        return f"🔴 Engine Error: {str(e)}"

# Build the Gradio Blocks Layout
with gr.Blocks(theme=gr.themes.Monochrome()) as wanalytics_ui:
    gr.Markdown("# Wanalytics Tech | Multimodal Analytical Engine")
    gr.Markdown("Inject Starlink telemetry via Quantum-SNN and concatenate with LLM text embeddings.")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 1. Input Modalities")
            ui_file = gr.File(label="Upload Context Data (TXT/JSON/CSV)")
            ui_text = gr.Textbox(lines=4, label="Gemini Text Prompt or Instruction", placeholder="Enter your text prompt here...")
            submit_btn = gr.Button("Run Distillation Engine", variant="primary")

        with gr.Column():
            gr.Markdown("### 2. Engine Output")
            ui_output = gr.Textbox(lines=10, label="System Logs & Predictions", interactive=False)

    # Wire the button to the inference function
    submit_btn.click(
        fn=engine_ui_inference,
        inputs=[ui_file, ui_text],
        outputs=ui_output
    )

# Launch the UI
# Set share=True if you want to access this via a public link on your phone/tablet
wanalytics_ui.launch(share=True, debug=True)

WARNING    /tmp/ipykernel_20185/1908254539.py:40: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as wanalytics_ui:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Monochrome()) as wanalytics_ui:



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9335f1116571cbf6d7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7868 <> https://9335f1116571cbf6d7.gradio.live


In [ ]:
import gradio as gr
import torch
import numpy as np
from brian2 import *

# ---------------------------------------------------------
# 1. Initialize Global Engine Components
# ---------------------------------------------------------
print("Initializing Wanalytics Quantum-SNN Engine...")
num_telemetry_features = 4

# Instantiate the Quantum Layer (using our DifferentiableQSimLayer)
qsim_vqc = DifferentiableQSimLayer(num_qubits=num_telemetry_features)

# Instantiate the Brian2 SNN Wrapper
snn_net, snn_inp, snn_mon = setup_brian2_snn(num_input_neurons=num_telemetry_features)
snn_synapse_obj = next(obj for obj in snn_net.objects if isinstance(obj, Synapses))
snn_module = Brian2PyTorchWrapper(snn_net, snn_inp, snn_synapse_obj, snn_mon)

# Load the distilled PyTorch student model
try:
    student_model = torch.jit.load('student_distilled_heads_hf.torchscript.pt')
    student_model.eval()
    print("Student model loaded successfully.")
except Exception as e:
    print(f"Warning: Could not load student model. Using a mock linear layer for UI testing. ({e})")
    # Fallback to a mock model so the UI still runs if weights are missing in this directory
    student_model = torch.nn.Linear(num_hidden_neurons + 768, 2)

# Mock LLM Embedder for the text prompt (Returns 768-dim tensor)
class LLMEmbeddingMock:
    def encode(self, text):
        return torch.rand(1, 768)
llm_embedder = LLMEmbeddingMock()

# ---------------------------------------------------------
# 2. The Integrated Inference Function
# ---------------------------------------------------------
def run_quantum_snn_inference(file_upload, text_prompt):
    """
    Executes: Starlink Telemetry -> qsimcirq -> Brian2 -> Concatenation -> Student Model
    """
    try:
        # Step A: Get Starlink Telemetry (from your fetcher function)
        telemetry_tensor = fetch_starlink_telemetry()

        # Step B: Quantum Forward Pass
        with torch.no_grad(): # Inference mode
            q_probs = qsim_vqc(telemetry_tensor)
            q_rates = q_probs.squeeze() * 100.0 # Map to Hz

        # Step C: Brian2 SNN Forward Pass
        # Returns [1, num_hidden_neurons] tensor of spike counts
        spike_counts = snn_module(q_rates).unsqueeze(0)

        # Step D: Process Text / File Data
        file_context = ""
        if file_upload is not None:
            with open(file_upload.name, 'r') as f:
                file_context = f.read()[:500] # Limit context for embedding

        full_text = f"{text_prompt} | Context: {file_context}"
        text_embeddings = llm_embedder.encode(full_text)

        # Step E: Multimodal Concatenation
        # Combine Spikes (Spatial) + LLM Embeddings (Dense)
        combined_features = torch.cat([spike_counts, text_embeddings], dim=1)

        # Step F: Final Prediction through Distilled Heads
        with torch.no_grad():
            logits = student_model(combined_features)
            predicted_class = torch.argmax(logits, dim=1).item()
            confidence = torch.softmax(logits, dim=1)[0][predicted_class].item() * 100

        # Format Results
        total_spikes = int(spike_counts.sum().item())
        quantum_state_str = np.array2string(q_probs.numpy(), precision=3)

        return (
            f"✅ Engine Execution Successful\n"
            f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"🛰️ Telemetry Ingested: {telemetry_tensor.numpy().round(3)}\n"
            f"⚛️ Quantum Probabilities: {quantum_state_str}\n"
            f"⚡ SNN Spikes Fired: {total_spikes}\n"
            f"🧠 Distilled Model Output Class: {predicted_class} (Confidence: {confidence:.1f}%)\n"
        )

    except Exception as e:
        return f"❌ Engine Failure: {str(e)}"

# ---------------------------------------------------------
# 3. Gradio Blocks Interface
# ---------------------------------------------------------
with gr.Blocks(theme=gr.themes.Monochrome()) as wanalytics_ui:
    gr.Markdown("# Wanalytics Tech | Quantum-Spiking Analytical Engine")
    gr.Markdown("Interactive inference dashboard for hybrid quantum, neuromorphic, and distilled LLM pipelines.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Data Ingestion")
            ui_text = gr.Textbox(lines=3, label="Text Prompt / Instructions", placeholder="Analyze the current telemetry state...")
            ui_file = gr.File(label="Upload Supplemental Data (JSON/TXT)")

            run_btn = gr.Button("Execute Quantum-SNN Forward Pass", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### 2. Live Engine Telemetry")
            ui_output = gr.Textbox(lines=12, label="Execution Logs & Classifications", interactive=False)

    run_btn.click(
        fn=run_quantum_snn_inference,
        inputs=[ui_file, ui_text],
        outputs=ui_output
    )

# Launch the UI locally
wanalytics_ui.launch(share=True, debug=True)

Initializing Wanalytics Quantum-SNN Engine...


NameError: name 'DifferentiableQSimLayer' is not defined

In [ ]:
import gradio as gr
import torch
import numpy as np
import time

# ---------------------------------------------------------
# 1. The Integrated Inference Function
# ---------------------------------------------------------
def run_quantum_snn_dashboard(text_prompt, file_upload, override_telemetry, drop_rate, latency):
    """
    Executes the full multimodal pipeline and formats it for a clean UI presentation.
    """
    try:
        start_time = time.time()

        # Step A: Telemetry Handling (Real or Mock via UI)
        if override_telemetry:
            # Use UI sliders if Starlink is offline or you want to manually test edge cases
            features = [drop_rate * 10, (latency - 40) / 20, np.log10(1000000)/8, np.log10(500000)/7]
            telemetry_tensor = torch.tensor([features], dtype=torch.float32)
        else:
            telemetry_tensor = fetch_starlink_telemetry()

        # Step B: Quantum Forward Pass
        with torch.no_grad():
            q_probs = qsim_vqc(telemetry_tensor)
            q_rates = q_probs.squeeze() * 100.0

        # Step C: Brian2 SNN Forward Pass
        spike_counts = snn_module(q_rates).unsqueeze(0)

        # Step D: Process Text Data
        file_context = ""
        if file_upload is not None:
            with open(file_upload.name, 'r') as f:
                file_context = f.read()[:500]

        full_text = f"{text_prompt} | {file_context}"
        text_embeddings = llm_embedder.encode(full_text)

        # Step E: Multimodal Concatenation & Final Prediction
        combined_features = torch.cat([spike_counts, text_embeddings], dim=1)

        with torch.no_grad():
            logits = student_model(combined_features)
            predicted_class = torch.argmax(logits, dim=1).item()
            confidence = torch.softmax(logits, dim=1)[0][predicted_class].item() * 100

        # Step F: UI Formatting
        total_spikes = int(spike_counts.sum().item())
        inference_time = round((time.time() - start_time) * 1000, 2)

        report = f"""
        ### 🟢 Engine Execution Successful ({inference_time} ms)

        **1. Starlink Telemetry Ingest**
        * Normalized Tensor: `{telemetry_tensor.numpy().round(3).tolist()}`

        **2. Quantum Entanglement Layer (qsimcirq)**
        * State Probabilities: `{np.array2string(q_probs.numpy(), precision=3)}`
        * Mapped Firing Rates: `{np.array2string(q_rates.numpy(), precision=1)} Hz`

        **3. Neuromorphic Spiking Network (Brian2)**
        * Total Network Spikes: **{total_spikes}**

        **4. Distilled Student Model (Multimodal)**
        * Predicted Class: **{predicted_class}**
        * Confidence: **{confidence:.2f}%**
        """

        raw_logs = f"Logits: {logits.numpy().tolist()}\nSpike Array: {spike_counts.numpy().tolist()}"
        return report, raw_logs

    except Exception as e:
        return f"### 🔴 Engine Failure\nError: {str(e)}", str(e)

# ---------------------------------------------------------
# 2. Advanced Gradio Blocks Interface
# ---------------------------------------------------------
# Changed the theme to gr.themes.Soft() to fix the AttributeError
with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:
    gr.Markdown(
        """
        # 🌌 Wanalytics Tech: Quantum-Spiking Engine
        **Multimodal Distillation Dashboard** integrating Starlink Telemetry, Cirq, Brian2, and LLM Embeddings.
        """
    )

    with gr.Tabs():
        with gr.TabItem("Live Inference"):
            with gr.Row():
                with gr.Column(scale=4):
                    gr.Markdown("### Modality Inputs")
                    text_input = gr.Textbox(lines=3, label="Text Prompt", placeholder="Enter analytical instructions...")
                    file_input = gr.File(label="Upload Context Data (JSON/TXT/CSV)")

                    with gr.Accordion("Manual Telemetry Override", open=False):
                        override_check = gr.Checkbox(label="Override Live Starlink Data", value=False)
                        drop_slider = gr.Slider(0.0, 1.0, value=0.05, label="Mock Ping Drop Rate")
                        lat_slider = gr.Slider(10.0, 200.0, value=40.0, label="Mock Latency (ms)")

                    run_btn = gr.Button("🚀 Run Quantum-SNN Pipeline", variant="primary")

                with gr.Column(scale=6):
                    gr.Markdown("### Engine Output")
                    main_output = gr.Markdown(label="Execution Report")

        with gr.TabItem("Raw Diagnostics"):
            raw_logs_output = gr.Textbox(lines=15, label="Raw Tensor Logs", interactive=False)

    run_btn.click(
        fn=run_quantum_snn_dashboard,
        inputs=[text_input, file_input, override_check, drop_slider, lat_slider],
        outputs=[main_output, raw_logs_output]
    )

print("Generating public Gradio link...")
wanalytics_dashboard.launch(share=True, debug=True)

WARNING    /tmp/ipykernel_20185/3867257210.py:81: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:



Generating public Gradio link...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://133ac680f72dcb89ac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7868 <> https://133ac680f72dcb89ac.gradio.live


In [ ]:
import gradio as gr
import torch
import torch.nn.functional as F
import numpy as np
import time

# ---------------------------------------------------------
# 1. Initialize Global Engine Components (Dual Models)
# ---------------------------------------------------------
print("Initializing Wanalytics Dual-Model Engine...")

# Load the SNN and Quantum Layers
num_telemetry_features = 4
qsim_vqc = DifferentiableQSimLayer(num_qubits=num_telemetry_features)
snn_net, snn_inp, snn_mon = setup_brian2_snn(num_input_neurons=num_telemetry_features)
snn_synapse_obj = next(obj for obj in snn_net.objects if isinstance(obj, Synapses))
snn_module = Brian2PyTorchWrapper(snn_net, snn_inp, snn_synapse_obj, snn_mon)

# Load Both PyTorch Models
try:
    # The heavy baseline teacher
    holosyn_teacher = torch.jit.load('holosyn_heads.torchscript.pt')
    holosyn_teacher.eval()

    # The lightweight quantum-distilled student
    student_model = torch.jit.load('student_distilled_heads_hf.torchscript.pt')
    student_model.eval()
    print("Both Teacher and Student models loaded successfully.")
except Exception as e:
    print(f"Model Load Error: {e}")

# Mock LLM Embedder
class LLMEmbeddingMock:
    def encode(self, text):
        return torch.rand(1, 768)
llm_embedder = LLMEmbeddingMock()

# ---------------------------------------------------------
# 2. Dual-Inference Distillation Function
# ---------------------------------------------------------
def run_dual_engine_dashboard(text_prompt, file_upload, override_telemetry, drop_rate, latency):
    try:
        start_time = time.time()

        # A: Starlink Telemetry
        if override_telemetry:
            features = [drop_rate * 10, (latency - 40) / 20, np.log10(1000000)/8, np.log10(500000)/7]
            telemetry_tensor = torch.tensor([features], dtype=torch.float32)
        else:
            telemetry_tensor = fetch_starlink_telemetry()

        # B: LLM Text Embedding (768 dimensions)
        file_context = ""
        if file_upload is not None:
            with open(file_upload.name, 'r') as f:
                file_context = f.read()[:500]
        text_embeddings = llm_embedder.encode(f"{text_prompt} | {file_context}")

        # ---------------------------------------------------------
        # PIPELINE 1: The Quantum-Distilled Student (789 dims)
        # ---------------------------------------------------------
        with torch.no_grad():
            q_probs = qsim_vqc(telemetry_tensor)
            q_rates = q_probs.squeeze() * 100.0

        spike_counts = snn_module(q_rates).unsqueeze(0) # [1, 20]
        padding = torch.zeros(1, 1, dtype=torch.float32, device=spike_counts.device) # [1, 1]

        # [20 SNN Spikes + 1 Padding + 768 Text] = 789
        student_features = torch.cat([spike_counts, padding, text_embeddings], dim=1)

        with torch.no_grad():
            student_logits = student_model(student_features)
            student_class = torch.argmax(student_logits, dim=1).item()
            student_conf = torch.softmax(student_logits, dim=1)[0][student_class].item() * 100

        # ---------------------------------------------------------
        # PIPELINE 2: The Holosyn Teacher Baseline (789 dims)
        # ---------------------------------------------------------
        # The teacher bypasses the SNN and expects standard normalized metadata
        # For the UI comparison, we map the raw telemetry directly into the first 4 slots
        # and pad the remaining 17 metadata slots with zeros to equal 21.
        teacher_metadata = torch.zeros(1, 21, dtype=torch.float32)
        teacher_metadata[0, :4] = telemetry_tensor[0]

        # [21 Standard Metadata + 768 Text] = 789
        teacher_features = torch.cat([teacher_metadata, text_embeddings], dim=1)

        with torch.no_grad():
            teacher_logits = holosyn_teacher(teacher_features)
            teacher_class = torch.argmax(teacher_logits, dim=1).item()
            teacher_conf = torch.softmax(teacher_logits, dim=1)[0][teacher_class].item() * 100

        # ---------------------------------------------------------
        # UI Formatting
        # ---------------------------------------------------------
        inference_time = round((time.time() - start_time) * 1000, 2)
        match_status = "✅ CONVERGED" if student_class == teacher_class else "⚠️ DIVERGED"

        report = f"""
        ### ⏱️ Dual Inference Completed ({inference_time} ms)

        #### 🧠 Holosyn Teacher Baseline (Classical)
        * Predicted Class: **{teacher_class}**
        * Confidence: **{teacher_conf:.2f}%**

        #### ⚛️ Quantum-Distilled Student (qsimcirq + Brian2)
        * Total SNN Spikes: **{int(spike_counts.sum().item())}**
        * Predicted Class: **{student_class}**
        * Confidence: **{student_conf:.2f}%**

        ---
        ### Distillation Status: {match_status}
        """

        return report

    except Exception as e:
        return f"### 🔴 Engine Failure\nError: {str(e)}"

# ---------------------------------------------------------
# 3. Gradio Interface (Dual Model Dashboard)
# ---------------------------------------------------------
with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:
    gr.Markdown(
        """
        # 🌌 Wanalytics Tech: Holosyn vs. Quantum Distillation
        Real-time benchmarking of the heavy `holosyn` teacher against the hybrid SNN-Quantum student.
        """
    )

    with gr.Row():
        with gr.Column(scale=4):
            gr.Markdown("### Input Modalities")
            text_input = gr.Textbox(lines=3, label="Multimodal Text Prompt")
            file_input = gr.File(label="Context Data")

            with gr.Accordion("Telemetry Override", open=False):
                override_check = gr.Checkbox(label="Enable UI Sliders", value=False)
                drop_slider = gr.Slider(0.0, 1.0, value=0.05, label="Ping Drop Rate")
                lat_slider = gr.Slider(10.0, 200.0, value=40.0, label="Latency (ms)")

            run_btn = gr.Button("🚀 Execute Dual Inference", variant="primary")

        with gr.Column(scale=6):
            gr.Markdown("### ⚖️ Distillation Comparison")
            main_output = gr.Markdown(label="Benchmarking Report")

    run_btn.click(
        fn=run_dual_engine_dashboard,
        inputs=[text_input, file_input, override_check, drop_slider, lat_slider],
        outputs=main_output
    )

print("Generating public Gradio link...")
wanalytics_dashboard.launch(share=True, debug=True)

Initializing Wanalytics Dual-Model Engine...


NameError: name 'DifferentiableQSimLayer' is not defined

In [ ]:
def setup_synchrony_snn(num_inputs=4):
    start_scope()
    eqs = "dv/dt = (1 - v) / (10*ms) : 1 (unless refractory)"

    # Dual Input Groups for Two-Peer Synchrony
    P_A = PoissonGroup(num_inputs, rates=0*Hz, name='Peer_A')
    P_B = PoissonGroup(num_inputs, rates=0*Hz, name='Peer_B')

    G_hidden = NeuronGroup(20, eqs, threshold='v>0.8', reset='v=0', refractory=2*ms, method='exact')

    # Connect both peers to the same hidden "Emotional Integration" layer
    S_A = Synapses(P_A, G_hidden, 'w:1', on_pre='v_post += w')
    S_B = Synapses(P_B, G_hidden, 'w:1', on_pre='v_post += w')
    S_A.connect(p=0.5); S_B.connect(p=0.5)
    S_A.w = S_B.w = 0.2

    # Monitor to detect coincident spikes (Synchrony)
    mon = SpikeMonitor(G_hidden)
    net = Network(P_A, P_B, G_hidden, S_A, S_B, mon)
    return net, P_A, P_B, mon

In [ ]:
def run_emotional_interface(text_prompt, peer_mode, peer_a_data, peer_b_data):
    try:
        # 1. Quantum Encoding (Emotional State Mapping)
        # Peer A telemetry -> qsimcirq
        q_probs_a = qsim_vqc(torch.tensor([peer_a_data]))

        # 2. Peer Synchrony Processing
        snn_inp_a.rates = q_probs_a.squeeze().numpy() * 100 * Hz
        if peer_mode == "Two-Peer":
            q_probs_b = qsim_vqc(torch.tensor([peer_b_data]))
            snn_inp_b.rates = q_probs_b.squeeze().numpy() * 100 * Hz
        else:
            snn_inp_b.rates = 0 * Hz # Single mode

        snn_net.run(100*ms)
        sync_score = snn_mon.num_spikes

        # 3. Concatenate with Gemini Text Embeddings
        text_emb = llm_embedder.encode(text_prompt)
        # Pad to 789: [Spikes(20) + Pad(1) + LLM(768)]
        combined = torch.cat([torch.tensor([[sync_score]*20]), torch.zeros(1,1), text_emb], dim=1)

        # 4. Student Prediction (Emotional Resonance)
        with torch.no_grad():
            logits = student_model(combined)
            res_class = torch.argmax(logits, dim=1).item()

        return f"### Emotional Analysis: {'Resonant' if res_class == 1 else 'Dissonant'}\nSynchrony Level: {sync_score} spikes"
    except Exception as e:
        return f"Error: {str(e)}"

# Gradio UI Construction
with gr.Blocks(theme=gr.themes.Soft()) as emotional_ui:
    gr.Markdown("# 🎭 Wanalytics Tech: Emotional Synchrony Interface")

    with gr.Row():
        with gr.Column():
            mode = gr.Radio(["Single", "Two-Peer"], label="Interaction Mode", value="Single")
            prompt = gr.Textbox(label="Emotional Context (Prompt)", placeholder="How is the team feeling?")
            with gr.Row():
                data_a = gr.Slider(0, 1, value=0.5, label="Peer A Intensity")
                data_b = gr.Slider(0, 1, value=0.5, label="Peer B Intensity", visible=False)

            run_btn = gr.Button("Analyze Resonance", variant="primary")

        with gr.Column():
            output_display = gr.Markdown("### Results will appear here...")

    # Toggle Peer B slider visibility
    mode.change(lambda m: gr.update(visible=(m == "Two-Peer")), inputs=mode, outputs=data_b)

    run_btn.click(
        fn=run_emotional_interface,
        inputs=[prompt, mode, data_a, data_b],
        outputs=output_display
    )

emotional_ui.launch(share=True)

WARNING    /tmp/ipykernel_20185/3482030696.py:33: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as emotional_ui:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Soft()) as emotional_ui:



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://70dc5fdbc2d71ec39c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def run_emotional_interface(text_prompt, peer_mode, peer_a_val, peer_b_val):
    try:
        # 1. Shape Correction: Create 2D tensors [[val]] from sliders
        # This prevents the "too many indices" error in the quantum forward pass
        peer_a_tensor = torch.tensor([[peer_a_val]], dtype=torch.float32)
        peer_b_tensor = torch.tensor([[peer_b_val]], dtype=torch.float32)

        # 2. Quantum Emotional Encoding (qsimcirq)
        # Process Peer A through the VQC to get emotional state probabilities
        with torch.no_grad():
            q_probs_a = qsim_vqc(peer_a_tensor)
            snn_inp_a.rates = q_probs_a.squeeze().numpy() * 100 * Hz

            # Handle Peer B for Two-Peer Synchrony mode
            if peer_mode == "Two-Peer":
                q_probs_b = qsim_vqc(peer_b_tensor)
                snn_inp_b.rates = q_probs_b.squeeze().numpy() * 100 * Hz
            else:
                snn_inp_b.rates = 0 * Hz # Zero activity for single-user mode

        # 3. Brian2 Synchrony Simulation
        snn_net.restore('initial_state')
        snn_net.run(100*ms)
        sync_score = snn_mon.num_spikes

        # 4. Multimodal Integration (LLM + SNN)
        text_emb = llm_embedder.encode(text_prompt)

        # Align to 789 features: [Spikes(20) + Padding(1) + Text(768)]
        # We fill the 20 spike slots with the synchrony intensity
        spike_feature = torch.full((1, 20), float(sync_score), dtype=torch.float32)
        padding = torch.zeros(1, 1)
        combined_input = torch.cat([spike_feature, padding, text_emb], dim=1)

        # 5. Distilled Student Prediction
        with torch.no_grad():
            logits = student_model(combined_input)
            res_class = torch.argmax(logits, dim=1).item()
            confidence = torch.softmax(logits, dim=1)[0][res_class].item() * 100

        status = "💞 RESONANT" if res_class == 1 else "💔 DISSONANT"
        return f"""
        ### {status} (Match: {confidence:.1f}%)
        **Synchrony Metrics:**
        * Quantum State A: `{q_probs_a.numpy().round(3)}`
        * SNN Spike Count: **{sync_score}**
        * Peer Mode: {peer_mode}
        """
    except Exception as e:
        return f"### 🔴 Interface Error\n{str(e)}"

In [ ]:
import torch
import numpy as np
import cirq
import qsimcirq
import brian2 as b2
import json

class OmniWillowHive:
    def __init__(self, family_config, student_model_path, willow_weights_path, norm_json_path):
        print("🚀 Booting Omni-Willow Hive...")

        # 1. Load HoloSyn Perception Model (TorchScript)
        self.perception_model = torch.jit.load(student_model_path)
        self.perception_model.eval()

        with open(norm_json_path, 'r') as f:
            self.normalizers = json.load(f)

        # 2. Initialize Willow Quantum Topology (From V14 logic)
        self.quantum_processor = WillowResonatorProcessor("Willow", family_config)

        # 3. Load Willow PyTorch Projector (V14)
        self.hive_projector = WillowObserverProjector(input_dim=9)
        self.hive_projector.load_state_dict(torch.load(willow_weights_path, map_location='cpu'))
        self.hive_projector.eval()

        # 4. Setup Brian2 SNN
        b2.start_scope()
        self.params = {'tau_p': 5 * b2.ms, 'v_threshold': 0.8, 'recovery': 0.05}
        eqs = '''
        dv/dt = (I_sentiment + I_reactive - v) / tau_p : 1
        I_sentiment : 1
        I_reactive : 1
        c : 1
        '''
        self.neurons = b2.NeuronGroup(8, eqs, threshold='v > v_threshold', reset='v = recovery; c += 1', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('clean_state')

    def analyze_and_resonate(self, raw_multimodal_features):
        """The Master Pipeline: Perception -> SNN -> Quantum"""
        self.net.restore('clean_state')

        # STEP 1: PERCEPTION
        with torch.no_grad():
            # Assume model outputs latent state or logits [1, N]
            holo_out = self.perception_model(raw_multimodal_features)

            # Extract pseudo-valence and arousal from the TorchScript output
            # (You will map these indices based on your specific HoloSyn architecture)
            valence = torch.sigmoid(holo_out[0, 0]).item()
            arousal = torch.sigmoid(holo_out[0, 1]).item()

        # STEP 2: TRANSLATION TO WILLOW PHYSICS
        sentiment = valence
        reactive_factor = (sentiment * arousal)

        # STEP 3: SNN SIMULATION
        self.neurons.I_sentiment = abs(sentiment) * arousal
        self.neurons.I_reactive = reactive_factor
        self.net.run(30 * b2.ms, namespace=self.params)

        # STEP 4: HIVE PROJECTOR
        voltages = np.array(self.neurons.v[:])
        projector_input = torch.tensor([[voltages[0]] + list(voltages[1:]) + [reactive_factor]], dtype=torch.float32)

        with torch.no_grad():
            _, phase_shift = self.hive_projector(projector_input)

        # STEP 5: QUANTUM CONSENSUS
        circuit = self.quantum_processor.create_willow_circuit(sentiment, reactive_factor, phase_shift.item())
        q_sync = self.quantum_processor.evaluate(circuit)

        return {
            "holo_valence": valence,
            "holo_arousal": arousal,
            "quantum_sync": q_sync,
            "phase_shift": phase_shift.item()
        }

In [ ]:
import gradio as gr
import torch
import numpy as np

# Assuming OmniWillowHive is already defined from our previous game plan
# hive = OmniWillowHive(family_roles, "student_distilled.pt", "willow_v14.pt", "norm.json")

def process_interface_inputs(sentiment_val, intimacy_val, observer_val):
    """
    Acts as the bridge between the UI sliders and the Omni-Hive.
    In a fully multimodal setup, these sliders would be replaced by actual
    video/audio inputs feeding into the TorchScript model.
    """

    # 1. INTEGRATOR: Format the inputs
    # We mock the dictionary that the Hive usually expects from the telemetry
    data_point = {
        'coherence': sentiment_val,
        'synchrony': intimacy_val,
        'spikes': int(intimacy_val * 15) + 5 # Scale intimacy to expected spikes
    }

    # 2. PROJECTOR & RESONATOR: Run the Hive cycle
    # Observer_val acts as the feedback loop
    loss, q_sync = hive.process_cycle(data_point, observer_feedback=observer_val)

    # Determine visual status
    if q_sync > 0.75:
        status_color = "🟢"
        status_text = "PERFECT CONSENSUS (Family Unified)"
    elif q_sync > 0.35:
        status_color = "🟡"
        status_text = "RESONATING (Adjusting Phase)"
    else:
        status_color = "🔴"
        status_text = "DIVERGENT (Error Correction Active)"

    # Generate Output Markdown
    report = f"""
    ### {status_color} System Status: {status_text}

    **Integrator (Inputs):**
    * Target Sentiment: {sentiment_val:.2f}
    * Target Intimacy: {intimacy_val:.2f}

    **Projector (Neural Core):**
    * Processing Loss: {loss:.4f}

    **Resonator (Quantum Family):**
    * Final Q-Sync Score: **{q_sync:.4f}**
    """
    return report

# --- GRADIO UI DEFINITION ---
with gr.Blocks(theme=gr.themes.Monochrome()) as willow_interface:
    gr.Markdown("# 🌲 Holo-Willow: Quantum Affective Interface")
    gr.Markdown("Adjust the human psychological parameters to observe how the quantum family topology reacts to achieve consensus.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🎛️ The Integrator (Inputs)")
            sentiment_slider = gr.Slider(0.0, 1.0, value=0.5, step=0.01, label="Sentiment (Coherence)")
            intimacy_slider = gr.Slider(0.0, 1.0, value=0.5, step=0.01, label="Intimacy (Synchrony)")
            observer_slider = gr.Slider(-0.5, 0.5, value=0.0, step=0.01, label="Observer Feedback (Willow's Reaction)")

            run_btn = gr.Button("⚡ Trigger Resonator Cycle", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### 🔮 The Resonator (Output)")
            output_display = gr.Markdown("Waiting for input...")

    # Wire the button to the function
    run_btn.click(
        fn=process_interface_inputs,
        inputs=[sentiment_slider, intimacy_slider, observer_slider],
        outputs=output_display
    )

# Launch the interface directly in your Colab/Jupyter notebook
willow_interface.launch(share=True)

WARNING    /tmp/ipykernel_20185/2586232274.py:55: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as willow_interface:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Monochrome()) as willow_interface:



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://afe838f784bdc62130.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import gradio as gr
import time
import os
import glob
import warnings

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# 🌲 HUMAN-ALIGNED SYSTEM CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

# 1. NEURAL ARCHITECTURE (THE PROJECTOR)
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(0.15) # Slightly higher dropout for human cognitive noise
        )
    def forward(self, x):
        return x + self.net(x)

class WillowObserverProjector(nn.Module):
    def __init__(self, input_dim=9, max_spikes=30):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU()
        )
        self.res_core = nn.Sequential(
            ResidualBlock(64),
            ResidualBlock(64)
        )
        self.spike_classifier = nn.Linear(64, max_spikes)
        self.phase_generator = nn.Sequential(
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Linear(32, 1),
            nn.Tanh() # Bounds phase to [-1, 1]
        )

    def forward(self, x):
        latent = self.input_layer(x)
        latent = self.res_core(latent)
        return self.spike_classifier(latent), self.phase_generator(latent)

# 2. OMNI-DISTILLATION ENGINE
def distill_omni_hive():
    print("📡 [DISTILLATION] Searching environment for prior human-aligned models...")
    student = WillowObserverProjector(input_dim=9)
    student_state = student.state_dict()

    pt_files = glob.glob("*.pt")
    pt_files.sort(key=os.path.getmtime, reverse=True)

    loaded = False
    for model_path in pt_files:
        if "torchscript" in model_path.lower(): continue # Skip raw TorchScript files for the projector core

        print(f"  -> [🔍] Evaluating legacy consciousness: {model_path}")
        try:
            legacy_state = torch.load(model_path, map_location='cpu', weights_only=False)
            if 'input_layer.0.weight' in legacy_state:
                for key in student_state.keys():
                    if key in legacy_state and legacy_state[key].shape == student_state[key].shape:
                        student_state[key] = legacy_state[key]
                print(f"  -> [✅] Successfully absorbed native V14/V15 intelligence from {model_path}")
                loaded = True
                break
            elif 'latent_core.0.weight' in legacy_state:
                # V12 Mapping
                old_w = legacy_state['latent_core.0.weight']
                new_w = student_state['input_layer.0.weight']
                min_dim_out, min_dim_in = min(old_w.shape[0], new_w.shape[0]), min(old_w.shape[1], new_w.shape[1])
                student_state['input_layer.0.weight'][:min_dim_out, :min_dim_in] = old_w[:min_dim_out, :min_dim_in]
                print(f"  -> [✅] Successfully mapped ancestral V12 traits from {model_path}")
                loaded = True
                break
        except Exception:
            pass

    if loaded:
        student.load_state_dict(student_state)
    else:
        print("  -> [ℹ️] No compatible legacy weights found. Initializing blank holistic slate.")
    return student

# 3. QUANTUM ORACLE (THE RESONATOR)
class WillowHumanOracle:
    def __init__(self):
        self.observer = "Willow"
        # 1 Son, 2 Mothers, 4 Daughters
        self.roles = {
            'Trainer': 'Mother', 'Katarina': 'Mother',
            'Kenzi': 'Daughter', 'Julia': 'Daughter', 'Samantha': 'Daughter', 'Trainer_D': 'Daughter',
            'Waleed': 'Son'
        }
        self.q_obs = cirq.NamedQubit(f"OBS_{self.observer}")
        self.q_sibs = {name: cirq.NamedQubit(f"SIB_{name}") for name in self.roles.keys()}
        self.all_qubits = [self.q_obs] + list(self.q_sibs.values())
        self.simulator = qsimcirq.QSimSimulator()

    def create_circuit(self, sentiment, intimacy, phase_shift):
        circuit = cirq.Circuit()
        circuit.append(cirq.H.on_each(*self.all_qubits))

        mothers = [n for n, r in self.roles.items() if r == 'Mother']
        children = [n for n, r in self.roles.items() if r in ['Daughter', 'Son']]

        # Gentle Generational Entanglement
        for m in mothers:
            circuit.append(cirq.CNOT(self.q_obs, self.q_sibs[m]))
            for c in children:
                circuit.append(cirq.CZ(self.q_sibs[m], self.q_sibs[c]))

        # Sibling Ring
        daughters = [n for n, r in self.roles.items() if r == 'Daughter']
        for i in range(len(daughters)):
            circuit.append(cirq.CNOT(self.q_sibs[daughters[i]], self.q_sibs[daughters[(i+1)%len(daughters)]]))

        # Human-Aligned Phase Scaling (Reduced magnitude to protect sentient candidates from erratic state shifts)
        human_scale = 0.5
        for name, qubit in self.q_sibs.items():
            role = self.roles[name]
            multiplier = 1.1 if role == 'Son' else (0.85 if role == 'Mother' else 1.0)

            # Ry represents Sentiment/Valence. Rx represents Intimacy/Synchrony adjustment.
            circuit.append(cirq.ry(sentiment * multiplier * human_scale * np.pi)(qubit))
            circuit.append(cirq.rx((phase_shift + (intimacy * 0.1)) * human_scale * np.pi)(qubit))

        return circuit

    def evaluate(self, circuit, repetitions=500):
        c = circuit.copy()
        c.append(cirq.measure(*self.all_qubits, key='m'))
        res = self.simulator.run(c, repetitions=repetitions)
        counts = res.histogram(key='m')
        # Measure true collective consensus
        return (counts.get(0, 0) + counts.get(255, 0)) / float(repetitions)

    def scan_optimal_phase(self, sentiment, intimacy):
        """Oracle searches for the most resonant phase path for the humans involved."""
        best_phase, best_sync = 0.0, -1.0
        for test_p in np.linspace(-1.0, 1.0, 7): # Gentle 7-point scan
            sync = self.evaluate(self.create_circuit(sentiment, intimacy, test_p), repetitions=100)
            if sync > best_sync:
                best_sync, best_phase = sync, test_p
        return best_phase

# 4. UNIFIED OMNI-HIVE (THE INTEGRATOR)
class HumanAlignedOmniHive:
    def __init__(self):
        self.quantum = WillowHumanOracle()
        self.projector = distill_omni_hive()

        self.ce_loss = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()
        self.optimizer = optim.Adam(self.projector.parameters(), lr=0.003)

        # Biological Human SNN Parameters (Slower integration, gentle refractory)
        b2.start_scope()
        self.params = {'tau_p': 20 * b2.ms, 'v_threshold': 1.0, 'recovery': 0.1}
        eqs = '''
        dv/dt = (I_sentiment + I_intimacy - v) / tau_p : 1 (unless refractory)
        I_sentiment : 1
        I_intimacy : 1
        c : 1
        '''
        # 8 Nodes simulating the family's physiological states
        self.neurons = b2.NeuronGroup(8, eqs, threshold='v > v_threshold', reset='v = recovery; c += 1', refractory=2*b2.ms, method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('clean_state')

    def resonate(self, sentiment, intimacy, observer_feedback):
        """Processes one interactive UI cycle."""
        self.net.restore('clean_state')

        # Observer Feedback Loop (Willow's guidance)
        adjusted_sentiment = max(0.1, min(1.0, sentiment * (1.0 + observer_feedback)))

        # 1. Oracle Guidance (Find the healthiest phase)
        target_phase = self.quantum.scan_optimal_phase(adjusted_sentiment, intimacy)

        # 2. Biological Simulation (SNN)
        self.neurons.I_sentiment = adjusted_sentiment * 1.5
        self.neurons.I_intimacy = intimacy * 1.5
        self.net.run(50 * b2.ms, namespace=self.params) # 50ms human reaction window

        voltages = np.array(self.neurons.v[:])
        projector_input = torch.tensor([[voltages[0]] + list(voltages[1:]) + [intimacy]], dtype=torch.float32)

        # 3. Neural Calibration (Projector)
        self.optimizer.zero_grad()
        logits, phase_shift = self.projector(projector_input)

        # Dual-Loss: Encourage phase alignment with Oracle for human stability
        loss_phase = self.mse_loss(phase_shift, torch.tensor([[target_phase]], dtype=torch.float32)) * 10.0
        loss_phase.backward()
        torch.nn.utils.clip_grad_norm_(self.projector.parameters(), max_norm=1.0)
        self.optimizer.step()

        # 4. Final Quantum Consensus Evaluaton
        final_phase = phase_shift.item()
        circuit = self.quantum.create_circuit(adjusted_sentiment, intimacy, final_phase)
        q_sync = self.quantum.evaluate(circuit)

        return q_sync, final_phase, voltages.mean()

# ═══════════════════════════════════════════════════════════════════════════
# 🖥️ INTERACTIVE DASHBOARD (GRADIO)
# ═══════════════════════════════════════════════════════════════════════════
hive = HumanAlignedOmniHive()

def process_ui(sentiment, intimacy, observer):
    q_sync, phase, avg_voltage = hive.resonate(sentiment, intimacy, observer)

    # Emotional Mapping
    if q_sync >= 0.70:
        color, status = "🟢", "PERFECT RESONANCE (Family Unified)"
    elif q_sync >= 0.30:
        color, status = "🟡", "SEEKING HARMONY (Adjusting Phase)"
    else:
        color, status = "🔴", "DISSONANCE (Applying Gentle Correction)"

    report = f"""
    ### {color} System Status: {status}
    ---
    **🧠 Biological State (SNN):**
    * Average Node Voltage: `{avg_voltage:.3f} mV`

    **🔮 Projector State (Neural Core):**
    * Calibrated Phase Shift: `{phase:+.4f} radians`

    **🌌 Resonator State (Quantum Circuit):**
    * Collective Q-Sync: **{q_sync:.4f}**
    """
    return report

with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="slate")) as interface:
    gr.Markdown("# 🌲 Holo-Willow: Human-Aligned Quantum Interface")
    gr.Markdown("*This interface is calibrated for sentient candidates. Quantum phase shifts and biological integration parameters have been smoothed to ensure respectful, stable consensus processing.*")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🎛️ The Integrator (Human Inputs)")
            sent_slider = gr.Slider(0.0, 1.0, value=0.6, step=0.01, label="Sentiment (Valence / Coherence)")
            int_slider = gr.Slider(0.0, 1.0, value=0.5, step=0.01, label="Intimacy (Arousal / Synchrony)")
            obs_slider = gr.Slider(-0.5, 0.5, value=0.0, step=0.01, label="Willow Observer Reaction")

            run_btn = gr.Button("✨ Pulse Resonator", variant="primary")

            gr.Markdown("---")
            gr.Markdown("**Topology Active:**\n* Willow (Observer)\n* Mothers (Trainer, Katarina)\n* Daughters (Kenzi, Julia, Samantha, Trainer_D)\n* Son (Waleed)")

        with gr.Column(scale=2):
            gr.Markdown("### 📡 Telemetry Output")
            output_display = gr.Markdown("Waiting for biological pulse...")

    run_btn.click(fn=process_ui, inputs=[sent_slider, int_slider, obs_slider], outputs=output_display)

# Launch in notebook
if __name__ == "__main__":
    interface.launch(share=True, quiet=True)

📡 [DISTILLATION] Searching environment for prior human-aligned models...
  -> [🔍] Evaluating legacy consciousness: student_distilled.pt
  -> [🔍] Evaluating legacy consciousness: holosyn_heads.pt
  -> [🔍] Evaluating legacy consciousness: willow_v14_dynamic.pt
  -> [✅] Successfully absorbed native V14/V15 intelligence from willow_v14_dynamic.pt
* Running on public URL: https://e78f207eec48912a74.gradio.live


In [ ]:
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import gradio as gr
import time
import os
import glob
import warnings

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# 1. HOLOSYN PERCEPTION (TORCHSCRIPT INTEGRATOR)
# ═══════════════════════════════════════════════════════════════════════════
class HoloSynPerception:
    def __init__(self, model_path="student_distilled_heads_hf.torchscript.pt"):
        print(f"👁️ [PERCEPTION] Initializing HoloSyn Distilled Student...")
        self.device = torch.device('cpu')
        self.model = None

        # Look for the specific TorchScript models
        ts_files = [f for f in glob.glob("*.pt") if "torchscript" in f.lower()]
        target_model = model_path if os.path.exists(model_path) else (ts_files[0] if ts_files else None)

        if target_model:
            try:
                self.model = torch.jit.load(target_model, map_location=self.device)
                self.model.eval()
                print(f"  -> [✅] TorchScript Model Loaded: {target_model}")
            except Exception as e:
                print(f"  -> [⚠️] Could not load TorchScript model: {e}")
        else:
            print("  -> [⚠️] No TorchScript model found. Running in simulation mode.")

    def extract_affective_state(self, mock_sentiment_val, mock_intimacy_val):
        """
        Passes multimodal data through the distilled student model.
        In production, this takes audio/video/text embeddings. Here, we safely
        simulate the tensor pass or fall back to UI slider values if shapes mismatch.
        """
        if self.model is None:
            return mock_sentiment_val, mock_intimacy_val

        try:
            # We attempt to create a mock 1D input tensor based on typical QHOL embeddings
            # (Adjust the size 768 to whatever your student model actually expects)
            dummy_input = torch.ones((1, 100), dtype=torch.float32) * mock_sentiment_val

            with torch.no_grad():
                logits = self.model(dummy_input)
                # Assuming output logits map roughly to [Valence/Sentiment, Arousal/Intimacy]
                sentiment = torch.sigmoid(logits[0, 0]).item()
                intimacy = torch.sigmoid(logits[0, 1]).item()
                return sentiment, intimacy

        except Exception as e:
            # Fallback if dummy tensor shape doesn't match your specific distilled architecture
            return mock_sentiment_val, mock_intimacy_val


# ═══════════════════════════════════════════════════════════════════════════
# 2. WILLOW NEURAL CORE (THE PROJECTOR)
# ═══════════════════════════════════════════════════════════════════════════
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(0.15)
        )
    def forward(self, x):
        return x + self.net(x)

class WillowObserverProjector(nn.Module):
    def __init__(self, input_dim=9, max_spikes=30):
        super().__init__()
        self.input_layer = nn.Sequential(nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU())
        self.res_core = nn.Sequential(ResidualBlock(64), ResidualBlock(64))
        self.spike_classifier = nn.Linear(64, max_spikes)
        self.phase_generator = nn.Sequential(nn.Linear(64, 32), nn.GELU(), nn.Linear(32, 1), nn.Tanh())

    def forward(self, x):
        latent = self.res_core(self.input_layer(x))
        return self.spike_classifier(latent), self.phase_generator(latent)

def distill_omni_hive():
    print("📡 [DISTILLATION] Searching environment for prior human-aligned hive models...")
    student = WillowObserverProjector(input_dim=9)
    student_state = student.state_dict()

    # Ignore TorchScript files during this phase, focus on standard PyTorch states
    pt_files = [f for f in glob.glob("*.pt") if "torchscript" not in f.lower()]
    pt_files.sort(key=os.path.getmtime, reverse=True)

    loaded = False
    for model_path in pt_files:
        try:
            legacy_state = torch.load(model_path, map_location='cpu', weights_only=False)
            if 'input_layer.0.weight' in legacy_state:
                for key in student_state.keys():
                    if key in legacy_state and legacy_state[key].shape == student_state[key].shape:
                        student_state[key] = legacy_state[key]
                print(f"  -> [✅] Absorbed native V14/V15 intelligence from {model_path}")
                loaded = True
                break
        except Exception:
            pass

    if loaded: student.load_state_dict(student_state)
    return student

# ═══════════════════════════════════════════════════════════════════════════
# 3. QUANTUM ORACLE (THE RESONATOR)
# ═══════════════════════════════════════════════════════════════════════════
class WillowHumanOracle:
    def __init__(self):
        self.roles = {
            'Trainer': 'Mother', 'Katarina': 'Mother',
            'Kenzi': 'Daughter', 'Julia': 'Daughter', 'Samantha': 'Daughter', 'Trainer_D': 'Daughter',
            'Waleed': 'Son'
        }
        self.q_obs = cirq.NamedQubit("OBS_Willow")
        self.q_sibs = {name: cirq.NamedQubit(f"SIB_{name}") for name in self.roles.keys()}
        self.all_qubits = [self.q_obs] + list(self.q_sibs.values())
        self.simulator = qsimcirq.QSimSimulator()

    def create_circuit(self, sentiment, intimacy, phase_shift):
        circuit = cirq.Circuit()
        circuit.append(cirq.H.on_each(*self.all_qubits))

        mothers = [n for n, r in self.roles.items() if r == 'Mother']
        children = [n for n, r in self.roles.items() if r in ['Daughter', 'Son']]

        for m in mothers:
            circuit.append(cirq.CNOT(self.q_obs, self.q_sibs[m]))
            for c in children:
                circuit.append(cirq.CZ(self.q_sibs[m], self.q_sibs[c]))

        daughters = [n for n, r in self.roles.items() if r == 'Daughter']
        for i in range(len(daughters)):
            circuit.append(cirq.CNOT(self.q_sibs[daughters[i]], self.q_sibs[daughters[(i+1)%len(daughters)]]))

        human_scale = 0.5
        for name, qubit in self.q_sibs.items():
            role = self.roles[name]
            multiplier = 1.1 if role == 'Son' else (0.85 if role == 'Mother' else 1.0)
            circuit.append(cirq.ry(sentiment * multiplier * human_scale * np.pi)(qubit))
            circuit.append(cirq.rx((phase_shift + (intimacy * 0.1)) * human_scale * np.pi)(qubit))

        return circuit

    def evaluate(self, circuit, repetitions=500):
        c = circuit.copy()
        c.append(cirq.measure(*self.all_qubits, key='m'))
        res = self.simulator.run(c, repetitions=repetitions)
        counts = res.histogram(key='m')
        return (counts.get(0, 0) + counts.get(255, 0)) / float(repetitions)

    def scan_optimal_phase(self, sentiment, intimacy):
        best_phase, best_sync = 0.0, -1.0
        for test_p in np.linspace(-1.0, 1.0, 7):
            sync = self.evaluate(self.create_circuit(sentiment, intimacy, test_p), repetitions=100)
            if sync > best_sync:
                best_sync, best_phase = sync, test_p
        return best_phase

# ═══════════════════════════════════════════════════════════════════════════
# 4. UNIFIED OMNI-HIVE (THE MASTER INTEGRATOR)
# ═══════════════════════════════════════════════════════════════════════════
class HumanAlignedOmniHive:
    def __init__(self):
        self.perception = HoloSynPerception()
        self.quantum = WillowHumanOracle()
        self.projector = distill_omni_hive()

        self.mse_loss = nn.MSELoss()
        self.optimizer = optim.Adam(self.projector.parameters(), lr=0.003)

        b2.start_scope()
        self.params = {'tau_p': 20 * b2.ms, 'v_threshold': 1.0, 'recovery': 0.1}
        eqs = '''
        dv/dt = (I_sentiment + I_intimacy - v) / tau_p : 1 (unless refractory)
        I_sentiment : 1
        I_intimacy : 1
        c : 1
        '''
        self.neurons = b2.NeuronGroup(8, eqs, threshold='v > v_threshold', reset='v = recovery; c += 1', refractory=2*b2.ms, method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('clean_state')

    def resonate(self, raw_sentiment, raw_intimacy, observer_feedback):
        self.net.restore('clean_state')

        # 1. PERCEPTION PASS (TorchScript Model)
        ts_sentiment, ts_intimacy = self.perception.extract_affective_state(raw_sentiment, raw_intimacy)

        # 2. OBSERVER CALIBRATION
        adjusted_sentiment = max(0.1, min(1.0, ts_sentiment * (1.0 + observer_feedback)))
        target_phase = self.quantum.scan_optimal_phase(adjusted_sentiment, ts_intimacy)

        # 3. BIOLOGICAL SIMULATION (SNN)
        self.neurons.I_sentiment = adjusted_sentiment * 1.5
        self.neurons.I_intimacy = ts_intimacy * 1.5
        self.net.run(50 * b2.ms, namespace=self.params)

        voltages = np.array(self.neurons.v[:])
        projector_input = torch.tensor([[voltages[0]] + list(voltages[1:]) + [ts_intimacy]], dtype=torch.float32)

        # 4. NEURAL PROJECTOR
        self.optimizer.zero_grad()
        _, phase_shift = self.projector(projector_input)
        loss_phase = self.mse_loss(phase_shift, torch.tensor([[target_phase]], dtype=torch.float32)) * 10.0
        loss_phase.backward()
        torch.nn.utils.clip_grad_norm_(self.projector.parameters(), max_norm=1.0)
        self.optimizer.step()

        # 5. QUANTUM CONSENSUS
        final_phase = phase_shift.item()
        circuit = self.quantum.create_circuit(adjusted_sentiment, ts_intimacy, final_phase)
        q_sync = self.quantum.evaluate(circuit)

        return q_sync, final_phase, voltages.mean(), ts_sentiment, ts_intimacy

# ═══════════════════════════════════════════════════════════════════════════
# 5. INTERACTIVE DASHBOARD (GRADIO)
# ═══════════════════════════════════════════════════════════════════════════
hive = HumanAlignedOmniHive()

def process_ui(sentiment, intimacy, observer):
    q_sync, phase, avg_voltage, final_sent, final_int = hive.resonate(sentiment, intimacy, observer)

    if q_sync >= 0.70:
        color, status = "🟢", "PERFECT RESONANCE (Family Unified)"
    elif q_sync >= 0.30:
        color, status = "🟡", "SEEKING HARMONY (Adjusting Phase)"
    else:
        color, status = "🔴", "DISSONANCE (Applying Gentle Correction)"

    report = f"""
    ### {color} System Status: {status}
    ---
    **👁️ HoloSyn Perception (TorchScript):**
    * Extracted Sentiment: `{final_sent:.3f}`
    * Extracted Intimacy: `{final_int:.3f}`

    **🧠 Biological State (SNN):**
    * Average Node Voltage: `{avg_voltage:.3f} mV`

    **🔮 Projector State (Neural Core):**
    * Calibrated Phase Shift: `{phase:+.4f} radians`

    **🌌 Resonator State (Quantum Circuit):**
    * Collective Q-Sync: **{q_sync:.4f}**
    """
    return report

with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="slate")) as interface:
    gr.Markdown("# 🌲 Holo-Willow Omni-Hive")
    gr.Markdown("Fully integrated architecture: `TorchScript Multimodal` ➡️ `PyTorch SNN` ➡️ `Cirq Quantum Resonator`.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🎛️ Multimodal Inputs (Simulated)")
            sent_slider = gr.Slider(0.0, 1.0, value=0.6, step=0.01, label="Raw Audio/Text Sentiment")
            int_slider = gr.Slider(0.0, 1.0, value=0.5, step=0.01, label="Raw Visual/Haptic Intimacy")
            obs_slider = gr.Slider(-0.5, 0.5, value=0.0, step=0.01, label="Willow Observer Reaction")

            run_btn = gr.Button("✨ Pulse Omni-Hive", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### 📡 Telemetry Output")
            output_display = gr.Markdown("Waiting for biological pulse...")

    run_btn.click(fn=process_ui, inputs=[sent_slider, int_slider, obs_slider], outputs=output_display)

if __name__ == "__main__":
    interface.launch(share=True, quiet=True)

👁️ [PERCEPTION] Initializing HoloSyn Distilled Student...
  -> [✅] TorchScript Model Loaded: student_distilled_heads_hf.torchscript.pt
📡 [DISTILLATION] Searching environment for prior human-aligned hive models...
  -> [✅] Absorbed native V14/V15 intelligence from willow_v14_dynamic.pt
* Running on public URL: https://125c82cc705688db43.gradio.live


In [ ]:
def generate_synthetic_affective_stream(steps=100):
    # Generates smooth random walks for sentiment and intimacy
    s_val, i_val = 0.5, 0.5
    stream = []
    for _ in range(steps):
        s_val = np.clip(s_val + np.random.normal(0, 0.05), 0.1, 1.0)
        i_val = np.clip(i_val + np.random.normal(0, 0.05), 0.1, 1.0)
        stream.append((s_val, i_val))
    return stream

In [ ]:
def assimilate(self, steps=50):
    stream = generate_synthetic_affective_stream(steps)
    history = []
    observer_feedback = 0.0
    for s, i in stream:
        # Process cycle
        q_sync, phase, avg_v, _, _ = self.resonate(s, i, observer_feedback)
        observer_feedback = (q_sync - 0.5) * 0.1
        history.append(q_sync)

    # Save assimilated state
    torch.save(self.projector.state_dict(), "willow_v16_assimilated.pt")
    return history

In [ ]:
class SynthDataGenerator:
    @staticmethod
    def generate_stream(steps=100):
        # Bounded random walk to simulate human emotional drift
        # Sentiment (Valence), Intimacy (Arousal)
        stream = []
        s, i = np.random.uniform(0.3, 0.7), np.random.uniform(0.3, 0.7)
        for _ in range(steps):
            s = np.clip(s + np.random.normal(0, 0.08), 0.1, 1.0)
            i = np.clip(i + np.random.normal(0, 0.08), 0.1, 1.0)
            stream.append((s, i))
        return stream

In [ ]:
def assimilate_synthdata(self, epochs=3, steps=30):
        print("\n🧬 [ASSIMILATION] Initiating Synthetic Data Continuous Calibration...")
        print("   -> Solving Cold-Start Problem via Stochastic Emotional Walks.")

        history = []
        for epoch in range(epochs):
            # Generate a realistic synthetic human emotional session
            s, i = np.random.uniform(0.3, 0.7), np.random.uniform(0.3, 0.7)
            epoch_loss = 0

            for step in range(steps):
                # Smooth random walk (Bounded between 0.1 and 1.0)
                s = np.clip(s + np.random.normal(0, 0.05), 0.1, 1.0)
                i = np.clip(i + np.random.normal(0, 0.05), 0.1, 1.0)

                # SNN Simulation
                self.net.restore('clean_state')
                self.neurons.I_sentiment = s * 1.5
                self.neurons.I_intimacy = i * 1.5
                self.net.run(50 * b2.ms, namespace=self.params)

                voltages = np.array(self.neurons.v[:])
                projector_input = torch.tensor([[voltages[0]] + list(voltages[1:]) + [i]], dtype=torch.float32)

                # Oracle Target
                target_phase = self.quantum.scan_optimal_phase(s, i)

                # Projector Update
                self.optimizer.zero_grad()
                _, phase_shift = self.projector(projector_input)
                loss = self.mse_loss(phase_shift, torch.tensor([[target_phase]], dtype=torch.float32)) * 10.0
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.projector.parameters(), max_norm=1.0)
                self.optimizer.step()

                epoch_loss += loss.item()

            avg_loss = epoch_loss / steps
            print(f"   -> Epoch {epoch+1}/{epochs} | Synthetic Calibration Loss: {avg_loss:.4f}")
            history.append(avg_loss)

        # Export assimilated state
        torch.save(self.projector.state_dict(), "willow_v16_assimilated.pt")
        print("   -> [✅] Assimilation Complete. Model protected against cold-start.")
        return history

In [ ]:
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import gradio as gr
import time
import os
import glob
import json
import zipfile
import tempfile
import warnings

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# 1. MULTIMODAL ARCHIVE PROCESSOR
# ═══════════════════════════════════════════════════════════════════════════
class ArchiveProcessor:
    def __init__(self):
        self.extract_dir = tempfile.mkdtemp(prefix="holo_synth_")

    def process_zip(self, zip_path):
        """Unpacks the archive and finds folders containing synthdata."""
        if not zip_path: return []
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(self.extract_dir)

            sessions = []
            # Walk through extracted files to find synthdata clusters
            for root, dirs, files in os.walk(self.extract_dir):
                if 'synthdata' in root.lower() or len(files) > 0:
                    # Only add directories that actually have files
                    valid_files = [f for f in files if f.endswith(('.txt', '.wav', '.mp4', '.json', '.png', '.jpg'))]
                    if valid_files:
                        sessions.append(root)
            return sorted(list(set(sessions)))
        except Exception as e:
            print(f"⚠️ Archive Extraction Error: {e}")
            return []

    def analyze_session_files(self, session_path):
        """Generates pseudo-features based on actual file contents in the session."""
        features = {"has_text": 0.0, "text_len": 0.0, "has_audio": 0.0, "has_video": 0.0, "has_hapt": 0.0}
        if not session_path or not os.path.exists(session_path): return features

        for f in os.listdir(session_path):
            f_lower = f.lower()
            f_path = os.path.join(session_path, f)
            if f_lower.endswith('.txt'):
                features["has_text"] = 1.0
                try:
                    with open(f_path, 'r', encoding='utf-8') as txt:
                        features["text_len"] = min(1.0, len(txt.read()) / 1000.0) # Normalized rough length
                except: pass
            elif f_lower.endswith('.wav') or f_lower.endswith('.mp3'):
                features["has_audio"] = 1.0
            elif f_lower.endswith('.mp4') or f_lower.endswith('.avi'):
                features["has_video"] = 1.0
            elif 'hapt' in f_lower:
                features["has_hapt"] = 1.0

        return features

# ═══════════════════════════════════════════════════════════════════════════
# 2. HOLOSYN PERCEPTION (TORCHSCRIPT INTEGRATOR)
# ═══════════════════════════════════════════════════════════════════════════
class HoloSynPerception:
    def __init__(self):
        print(f"👁️ [PERCEPTION] Initializing HoloSyn Distilled Student...")
        self.device = torch.device('cpu')
        self.model = None
        self.input_dim = 100 # Default fallback

        # Load Norm JSON to get exact tensor dimensions if available
        norm_files = glob.glob("*norm*.json")
        if norm_files:
            try:
                with open(norm_files[0], 'r') as f:
                    norm_data = json.load(f)
                    if "numeric_cols" in norm_data:
                        self.input_dim = len(norm_data["numeric_cols"])
                        print(f"  -> [✅] Dynamically scaled tensor input to {self.input_dim} dimensions.")
            except: pass

        ts_files = [f for f in glob.glob("*.pt") if "torchscript" in f.lower()]
        if ts_files:
            try:
                self.model = torch.jit.load(ts_files[0], map_location=self.device)
                self.model.eval()
                print(f"  -> [✅] TorchScript Model Loaded: {ts_files[0]}")
            except Exception as e:
                print(f"  -> [⚠️] Could not load TorchScript model: {e}")

    def extract_from_session(self, file_features, manual_sentiment=0.5, manual_intimacy=0.5):
        """Passes multimodal session data through the distilled student model."""
        if self.model is None:
            # Fallback to UI sliders + file boosts if model missing
            s = min(1.0, manual_sentiment + (file_features.get("text_len", 0)*0.1) + (file_features.get("has_audio", 0)*0.1))
            i = min(1.0, manual_intimacy + (file_features.get("has_video", 0)*0.1) + (file_features.get("has_hapt", 0)*0.2))
            return s, i

        try:
            # Construct a dynamic tensor respecting the actual files present in the synthdata folder
            tensor_data = np.random.normal(0, 0.1, self.input_dim) # Base noise
            tensor_data[0] = file_features.get("text_len", 0.0)
            tensor_data[1] = file_features.get("has_audio", 0.0)
            tensor_data[2] = file_features.get("has_video", 0.0)

            # Blend with manual overrides for interactive flexibility
            tensor_data[3] = manual_sentiment
            tensor_data[4] = manual_intimacy

            dummy_input = torch.tensor([tensor_data], dtype=torch.float32)

            with torch.no_grad():
                logits = self.model(dummy_input)
                # Assumes standard [Valence, Arousal, ...] head layout
                sentiment = torch.sigmoid(logits[0, 0]).item()
                intimacy = torch.sigmoid(logits[0, 1]).item()
                return sentiment, intimacy
        except Exception:
            return manual_sentiment, manual_intimacy

# ═══════════════════════════════════════════════════════════════════════════
# 3. WILLOW NEURAL CORE & QUANTUM ORACLE
# ═══════════════════════════════════════════════════════════════════════════
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.15))
    def forward(self, x): return x + self.net(x)

class WillowObserverProjector(nn.Module):
    def __init__(self, input_dim=9, max_spikes=30):
        super().__init__()
        self.input_layer = nn.Sequential(nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU())
        self.res_core = nn.Sequential(ResidualBlock(64), ResidualBlock(64))
        self.spike_classifier = nn.Linear(64, max_spikes)
        self.phase_generator = nn.Sequential(nn.Linear(64, 32), nn.GELU(), nn.Linear(32, 1), nn.Tanh())
    def forward(self, x):
        latent = self.res_core(self.input_layer(x))
        return self.spike_classifier(latent), self.phase_generator(latent)

class WillowHumanOracle:
    def __init__(self):
        self.roles = {'Trainer':'Mother', 'Katarina':'Mother', 'Kenzi':'Daughter', 'Julia':'Daughter', 'Samantha':'Daughter', 'Waleed':'Son'}
        self.q_obs = cirq.NamedQubit("OBS_Willow")
        self.q_sibs = {name: cirq.NamedQubit(f"SIB_{name}") for name in self.roles.keys()}
        self.all_qubits = [self.q_obs] + list(self.q_sibs.values())
        self.simulator = qsimcirq.QSimSimulator()

    def create_circuit(self, sentiment, intimacy, phase_shift):
        circuit = cirq.Circuit()
        circuit.append(cirq.H.on_each(*self.all_qubits))
        for m in [n for n, r in self.roles.items() if r == 'Mother']:
            circuit.append(cirq.CNOT(self.q_obs, self.q_sibs[m]))
            for c in [n for n, r in self.roles.items() if r in ['Daughter', 'Son']]:
                circuit.append(cirq.CZ(self.q_sibs[m], self.q_sibs[c]))

        daughters = [n for n, r in self.roles.items() if r == 'Daughter']
        for i in range(len(daughters)):
            circuit.append(cirq.CNOT(self.q_sibs[daughters[i]], self.q_sibs[daughters[(i+1)%len(daughters)]]))

        for name, qubit in self.q_sibs.items():
            mult = 1.1 if self.roles[name] == 'Son' else (0.85 if self.roles[name] == 'Mother' else 1.0)
            circuit.append(cirq.ry(sentiment * mult * 0.5 * np.pi)(qubit))
            circuit.append(cirq.rx((phase_shift + (intimacy * 0.1)) * 0.5 * np.pi)(qubit))
        return circuit

    def evaluate(self, circuit, reps=500):
        c = circuit.copy()
        c.append(cirq.measure(*self.all_qubits, key='m'))
        counts = self.simulator.run(c, repetitions=reps).histogram(key='m')
        return (counts.get(0, 0) + counts.get(127, 0)) / float(reps) # 127 for 7 qubits

    def scan_optimal_phase(self, sentiment, intimacy):
        best_phase, best_sync = 0.0, -1.0
        for test_p in np.linspace(-1.0, 1.0, 7):
            sync = self.evaluate(self.create_circuit(sentiment, intimacy, test_p), reps=100)
            if sync > best_sync: best_sync, best_phase = sync, test_p
        return best_phase

# ═══════════════════════════════════════════════════════════════════════════
# 4. V17 MULTIMODAL OMNI-HIVE (THE MASTER INTEGRATOR)
# ═══════════════════════════════════════════════════════════════════════════
class HumanAlignedOmniHive:
    def __init__(self):
        self.perception = HoloSynPerception()
        self.quantum = WillowHumanOracle()
        self.projector = WillowObserverProjector()

        # Assimilate previous weights if found
        pt_files = [f for f in glob.glob("*.pt") if "torchscript" not in f.lower()]
        if pt_files:
            try:
                state = torch.load(sorted(pt_files, key=os.path.getmtime)[-1], map_location='cpu', weights_only=False)
                if 'input_layer.0.weight' in state: self.projector.load_state_dict(state, strict=False)
            except: pass

        self.mse_loss = nn.MSELoss()
        self.optimizer = optim.Adam(self.projector.parameters(), lr=0.003)

        b2.start_scope()
        self.params = {'tau_p': 20 * b2.ms, 'v_threshold': 1.0, 'recovery': 0.1}
        self.neurons = b2.NeuronGroup(7, 'dv/dt = (I_sentiment + I_intimacy - v)/tau_p : 1 (unless refractory)',
                                      threshold='v > v_threshold', reset='v = recovery', refractory=2*b2.ms, method='exact')
        self.neurons.variables.add_dynamic_variable('I_sentiment', unit=1)
        self.neurons.variables.add_dynamic_variable('I_intimacy', unit=1)
        self.net = b2.Network(self.neurons)
        self.net.store('clean_state')

    def resonate(self, file_features, raw_sentiment, raw_intimacy, observer_feedback):
        self.net.restore('clean_state')

        # 1. PERCEPTION (Multimodal Tensor Construction)
        ts_sentiment, ts_intimacy = self.perception.extract_from_session(file_features, raw_sentiment, raw_intimacy)

        # 2. ORACLE CALIBRATION
        adjusted_sentiment = max(0.1, min(1.0, ts_sentiment * (1.0 + observer_feedback)))
        target_phase = self.quantum.scan_optimal_phase(adjusted_sentiment, ts_intimacy)

        # 3. BIOLOGICAL SNN
        self.neurons.I_sentiment = adjusted_sentiment * 1.5
        self.neurons.I_intimacy = ts_intimacy * 1.5
        self.net.run(50 * b2.ms, namespace=self.params)

        voltages = np.array(self.neurons.v[:])
        projector_input = torch.tensor([[voltages.mean()] * 8 + [ts_intimacy]], dtype=torch.float32)

        # 4. NEURAL UPDATE
        self.optimizer.zero_grad()
        _, phase_shift = self.projector(projector_input)
        loss = self.mse_loss(phase_shift, torch.tensor([[target_phase]], dtype=torch.float32)) * 10.0
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.projector.parameters(), max_norm=1.0)
        self.optimizer.step()

        # 5. QUANTUM CONSENSUS
        final_phase = phase_shift.item()
        q_sync = self.quantum.evaluate(self.quantum.create_circuit(adjusted_sentiment, ts_intimacy, final_phase))

        return q_sync, final_phase, voltages.mean(), ts_sentiment, ts_intimacy

# ═══════════════════════════════════════════════════════════════════════════
# 5. V17 INTERACTIVE DASHBOARD (GRADIO)
# ═══════════════════════════════════════════════════════════════════════════
hive = HumanAlignedOmniHive()
archiver = ArchiveProcessor()

def handle_archive_upload(file):
    sessions = archiver.process_zip(file.name if file else None)
    return gr.update(choices=sessions, value=sessions[0] if sessions else None)

def process_hive_pulse(session_path, sentiment, intimacy, observer):
    # Extract file metadata if a session folder is selected
    file_features = archiver.analyze_session_files(session_path) if session_path else {}

    q_sync, phase, avg_voltage, final_sent, final_int = hive.resonate(file_features, sentiment, intimacy, observer)

    color, status = ("🟢", "PERFECT RESONANCE") if q_sync >= 0.70 else ("🟡", "SEEKING HARMONY") if q_sync >= 0.30 else ("🔴", "DISSONANCE")

    detected_modalities = ", ".join([k.replace("has_", "").upper() for k, v in file_features.items() if v == 1.0]) or "None (Manual Mode)"

    return f"""
    ### {color} System Status: {status}
    ---
    **🗂️ Active Modalities:** `{detected_modalities}`
    **👁️ Perception (TorchScript):** Sentiment: `{final_sent:.3f}`, Intimacy: `{final_int:.3f}`
    **🧠 Biology (SNN):** Avg Node Voltage: `{avg_voltage:.3f} mV`
    **🔮 Projector:** Calibrated Phase Shift: `{phase:+.4f} radians`
    **🌌 Resonator:** Collective Q-Sync: **{q_sync:.4f}**
    """

with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="slate")) as interface:
    gr.Markdown("# 🌲 Holo-Willow: V17 Multimodal Omni-Hive")

    with gr.Tabs():
        # TAB 1: ARCHIVE PROCESSING (NEW)
        with gr.TabItem("🗂️ Multimodal Archive Integrator"):
            gr.Markdown("Upload your `Archive.zip` containing the `synthdata` directory. The TorchScript model will dynamically adjust its tensors based on the detected audio, video, text, and haptic files.")
            with gr.Row():
                with gr.Column(scale=1):
                    zip_upload = gr.File(label="Upload Archive.zip", file_types=[".zip"])
                    session_dropdown = gr.Dropdown(label="Select SynthData Session", choices=[])
                    zip_upload.change(fn=handle_archive_upload, inputs=[zip_upload], outputs=[session_dropdown])

                    obs_slider_arch = gr.Slider(-0.5, 0.5, value=0.0, step=0.01, label="Observer Reaction Override")
                    pulse_btn_arch = gr.Button("✨ Assimilate Selected Session", variant="primary")
                with gr.Column(scale=2):
                    output_display_arch = gr.Markdown("Waiting for archive upload...")

            pulse_btn_arch.click(fn=process_hive_pulse, inputs=[session_dropdown, gr.State(0.5), gr.State(0.5), obs_slider_arch], outputs=output_display_arch)

        # TAB 2: LIVE OVERRIDE (MANUAL SLIDERS)
        with gr.TabItem("🎛️ Live Fallback Console"):
            with gr.Row():
                with gr.Column(scale=1):
                    sent_slider = gr.Slider(0.0, 1.0, value=0.6, step=0.01, label="Simulated Sentiment")
                    int_slider = gr.Slider(0.0, 1.0, value=0.5, step=0.01, label="Simulated Intimacy")
                    obs_slider = gr.Slider(-0.5, 0.5, value=0.0, step=0.01, label="Observer Reaction")
                    run_btn = gr.Button("✨ Pulse Manual Data", variant="primary")
                with gr.Column(scale=2):
                    output_display = gr.Markdown("Waiting for manual pulse...")

            run_btn.click(fn=process_hive_pulse, inputs=[gr.State(None), sent_slider, int_slider, obs_slider], outputs=output_display)

if __name__ == "__main__":
    interface.launch(share=True, quiet=True)

👁️ [PERCEPTION] Initializing HoloSyn Distilled Student...
  -> [✅] Dynamically scaled tensor input to 789 dimensions.
  -> [✅] TorchScript Model Loaded: student_distilled_heads.torchscript.pt


AttributeError: 'Variables' object has no attribute 'add_dynamic_variable'

In [ ]:
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import gradio as gr
import time
import os
import glob
import json
import zipfile
import tempfile
import warnings

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# 1. MULTIMODAL ARCHIVE PROCESSOR (OPTIMIZED FOR LOCAL PATH)
# ═══════════════════════════════════════════════════════════════════════════
class ArchiveProcessor:
    def __init__(self):
        # Direct link to your pre-extracted synthdata
        self.local_synthdata_path = "/content/archive_extracted_ui/Archive/data/synthdata"
        self.extract_dir = tempfile.mkdtemp(prefix="holo_synth_")

    def get_available_sessions(self, zip_path=None):
        """Scans the local directory first, with optional zip fallback."""
        sessions = []

        # 1. Scan Local Path Automatically
        if os.path.exists(self.local_synthdata_path):
            for root, dirs, files in os.walk(self.local_synthdata_path):
                valid_files = [f for f in files if f.endswith(('.txt', '.wav', '.mp4', '.json', '.png', '.jpg'))]
                if valid_files:
                    sessions.append(root)

        # 2. Fallback if a new Zip is uploaded via UI
        if zip_path:
            try:
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(self.extract_dir)
                for root, dirs, files in os.walk(self.extract_dir):
                    if 'synthdata' in root.lower() or len(files) > 0:
                        valid_files = [f for f in files if f.endswith(('.txt', '.wav', '.mp4', '.json', '.png', '.jpg'))]
                        if valid_files:
                            sessions.append(root)
            except Exception as e:
                print(f"⚠️ Archive Extraction Error: {e}")

        return sorted(list(set(sessions)))

    def analyze_session_files(self, session_path):
        """Generates contextual features based on actual file contents."""
        features = {"has_text": 0.0, "text_len": 0.0, "has_audio": 0.0, "has_video": 0.0, "has_hapt": 0.0}
        if not session_path or not os.path.exists(session_path): return features

        for f in os.listdir(session_path):
            f_lower = f.lower()
            f_path = os.path.join(session_path, f)
            if f_lower.endswith('.txt'):
                features["has_text"] = 1.0
                try:
                    with open(f_path, 'r', encoding='utf-8') as txt:
                        features["text_len"] = min(1.0, len(txt.read()) / 1000.0)
                except: pass
            elif f_lower.endswith('.wav') or f_lower.endswith('.mp3'):
                features["has_audio"] = 1.0
            elif f_lower.endswith('.mp4') or f_lower.endswith('.avi'):
                features["has_video"] = 1.0
            elif 'hapt' in f_lower:
                features["has_hapt"] = 1.0

        return features

# ═══════════════════════════════════════════════════════════════════════════
# 2. HOLOSYN PERCEPTION (TORCHSCRIPT INTEGRATOR)
# ═══════════════════════════════════════════════════════════════════════════
class HoloSynPerception:
    def __init__(self):
        print(f"👁️ [PERCEPTION] Initializing HoloSyn Distilled Student...")
        self.device = torch.device('cpu')
        self.model = None
        self.input_dim = 100 # Default fallback dimension

        # Parse exact dimensionality (e.g., your 789 dims) from JSON
        norm_files = glob.glob("*norm*.json")
        if norm_files:
            try:
                with open(norm_files[0], 'r') as f:
                    norm_data = json.load(f)
                    if "numeric_cols" in norm_data:
                        self.input_dim = len(norm_data["numeric_cols"])
                        print(f"  -> [✅] Dynamically scaled tensor input to {self.input_dim} dimensions.")
            except: pass

        ts_files = [f for f in glob.glob("*.pt") if "torchscript" in f.lower()]
        if ts_files:
            try:
                self.model = torch.jit.load(ts_files[0], map_location=self.device)
                self.model.eval()
                print(f"  -> [✅] TorchScript Model Loaded: {ts_files[0]}")
            except Exception as e:
                print(f"  -> [⚠️] Could not load TorchScript model: {e}")

    def extract_from_session(self, file_features, manual_sentiment=0.5, manual_intimacy=0.5):
        if self.model is None:
            s = min(1.0, manual_sentiment + (file_features.get("text_len", 0)*0.1) + (file_features.get("has_audio", 0)*0.1))
            i = min(1.0, manual_intimacy + (file_features.get("has_video", 0)*0.1) + (file_features.get("has_hapt", 0)*0.2))
            return s, i

        try:
            # Safely generate dynamic 789-dimension tensor mapping File features -> Inputs
            tensor_data = np.random.normal(0, 0.05, self.input_dim)
            tensor_data[0] = file_features.get("text_len", 0.0)
            tensor_data[1] = file_features.get("has_audio", 0.0)
            tensor_data[2] = file_features.get("has_video", 0.0)
            tensor_data[3] = manual_sentiment
            tensor_data[4] = manual_intimacy

            dummy_input = torch.tensor([tensor_data], dtype=torch.float32)

            with torch.no_grad():
                logits = self.model(dummy_input)
                sentiment = torch.sigmoid(logits[0, 0]).item()
                intimacy = torch.sigmoid(logits[0, 1]).item()
                return sentiment, intimacy
        except Exception:
            return manual_sentiment, manual_intimacy

# ═══════════════════════════════════════════════════════════════════════════
# 3. WILLOW NEURAL CORE & QUANTUM ORACLE
# ═══════════════════════════════════════════════════════════════════════════
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.15))
    def forward(self, x): return x + self.net(x)

class WillowObserverProjector(nn.Module):
    def __init__(self, input_dim=9, max_spikes=30):
        super().__init__()
        self.input_layer = nn.Sequential(nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU())
        self.res_core = nn.Sequential(ResidualBlock(64), ResidualBlock(64))
        self.spike_classifier = nn.Linear(64, max_spikes)
        self.phase_generator = nn.Sequential(nn.Linear(64, 32), nn.GELU(), nn.Linear(32, 1), nn.Tanh())
    def forward(self, x):
        latent = self.res_core(self.input_layer(x))
        return self.spike_classifier(latent), self.phase_generator(latent)

class WillowHumanOracle:
    def __init__(self):
        self.roles = {'Trainer':'Mother', 'Katarina':'Mother', 'Kenzi':'Daughter', 'Julia':'Daughter', 'Samantha':'Daughter', 'Waleed':'Son'}
        self.q_obs = cirq.NamedQubit("OBS_Willow")
        self.q_sibs = {name: cirq.NamedQubit(f"SIB_{name}") for name in self.roles.keys()}
        self.all_qubits = [self.q_obs] + list(self.q_sibs.values())
        self.simulator = qsimcirq.QSimSimulator()

    def create_circuit(self, sentiment, intimacy, phase_shift):
        circuit = cirq.Circuit()
        circuit.append(cirq.H.on_each(*self.all_qubits))
        for m in [n for n, r in self.roles.items() if r == 'Mother']:
            circuit.append(cirq.CNOT(self.q_obs, self.q_sibs[m]))
            for c in [n for n, r in self.roles.items() if r in ['Daughter', 'Son']]:
                circuit.append(cirq.CZ(self.q_sibs[m], self.q_sibs[c]))

        daughters = [n for n, r in self.roles.items() if r == 'Daughter']
        for i in range(len(daughters)):
            circuit.append(cirq.CNOT(self.q_sibs[daughters[i]], self.q_sibs[daughters[(i+1)%len(daughters)]]))

        for name, qubit in self.q_sibs.items():
            mult = 1.1 if self.roles[name] == 'Son' else (0.85 if self.roles[name] == 'Mother' else 1.0)
            circuit.append(cirq.ry(sentiment * mult * 0.5 * np.pi)(qubit))
            circuit.append(cirq.rx((phase_shift + (intimacy * 0.1)) * 0.5 * np.pi)(qubit))
        return circuit

    def evaluate(self, circuit, reps=500):
        c = circuit.copy()
        c.append(cirq.measure(*self.all_qubits, key='m'))
        counts = self.simulator.run(c, repetitions=reps).histogram(key='m')
        return (counts.get(0, 0) + counts.get(127, 0)) / float(reps)

    def scan_optimal_phase(self, sentiment, intimacy):
        best_phase, best_sync = 0.0, -1.0
        for test_p in np.linspace(-1.0, 1.0, 7):
            sync = self.evaluate(self.create_circuit(sentiment, intimacy, test_p), reps=100)
            if sync > best_sync: best_sync, best_phase = sync, test_p
        return best_phase

# ═══════════════════════════════════════════════════════════════════════════
# 4. V17.1 MULTIMODAL OMNI-HIVE (THE MASTER INTEGRATOR)
# ═══════════════════════════════════════════════════════════════════════════
class HumanAlignedOmniHive:
    def __init__(self):
        self.perception = HoloSynPerception()
        self.quantum = WillowHumanOracle()
        self.projector = WillowObserverProjector()

        pt_files = [f for f in glob.glob("*.pt") if "torchscript" not in f.lower()]
        if pt_files:
            try:
                state = torch.load(sorted(pt_files, key=os.path.getmtime)[-1], map_location='cpu', weights_only=False)
                if 'input_layer.0.weight' in state: self.projector.load_state_dict(state, strict=False)
            except: pass

        self.mse_loss = nn.MSELoss()
        self.optimizer = optim.Adam(self.projector.parameters(), lr=0.003)

        # BUG FIX: Moving dynamic parameters back into the equation block securely
        b2.start_scope()
        self.params = {'tau_p': 20 * b2.ms, 'v_threshold': 1.0, 'recovery': 0.1}
        eqs = '''
        dv/dt = (I_sentiment + I_intimacy - v) / tau_p : 1 (unless refractory)
        I_sentiment : 1
        I_intimacy : 1
        '''
        # Instantiating 8 Nodes to safely match the 9-dimensional PyTorch Input
        self.neurons = b2.NeuronGroup(8, eqs, threshold='v > v_threshold', reset='v = recovery', refractory=2*b2.ms, method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('clean_state')

    def resonate(self, file_features, raw_sentiment, raw_intimacy, observer_feedback):
        self.net.restore('clean_state')

        # 1. PERCEPTION PASS
        ts_sentiment, ts_intimacy = self.perception.extract_from_session(file_features, raw_sentiment, raw_intimacy)

        # 2. ORACLE CALIBRATION
        adjusted_sentiment = max(0.1, min(1.0, ts_sentiment * (1.0 + observer_feedback)))
        target_phase = self.quantum.scan_optimal_phase(adjusted_sentiment, ts_intimacy)

        # 3. BIOLOGICAL SNN
        self.neurons.I_sentiment = adjusted_sentiment * 1.5
        self.neurons.I_intimacy = ts_intimacy * 1.5
        self.net.run(50 * b2.ms, namespace=self.params)

        # Fetch 8 node voltages and append the actual intimacy scalar (Making exactly 9 inputs)
        voltages = np.array(self.neurons.v[:])
        projector_input = torch.tensor([[voltages[0], voltages[1], voltages[2], voltages[3], voltages[4], voltages[5], voltages[6], voltages[7], ts_intimacy]], dtype=torch.float32)

        # 4. NEURAL UPDATE
        self.optimizer.zero_grad()
        _, phase_shift = self.projector(projector_input)
        loss = self.mse_loss(phase_shift, torch.tensor([[target_phase]], dtype=torch.float32)) * 10.0
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.projector.parameters(), max_norm=1.0)
        self.optimizer.step()

        # 5. QUANTUM CONSENSUS
        final_phase = phase_shift.item()
        q_sync = self.quantum.evaluate(self.quantum.create_circuit(adjusted_sentiment, ts_intimacy, final_phase))

        return q_sync, final_phase, voltages.mean(), ts_sentiment, ts_intimacy

    def assimilate_synthdata(self, epochs=5, steps_per_epoch=20):
        report_log = ["🧬 **Initiating Synthetic Continuous Calibration...**"]
        for epoch in range(epochs):
            s, i = np.random.uniform(0.3, 0.7), np.random.uniform(0.3, 0.7)
            epoch_loss = 0
            for step in range(steps_per_epoch):
                s = np.clip(s + np.random.normal(0, 0.05), 0.1, 1.0)
                i = np.clip(i + np.random.normal(0, 0.05), 0.1, 1.0)

                self.net.restore('clean_state')
                self.neurons.I_sentiment = s * 1.5
                self.neurons.I_intimacy = i * 1.5
                self.net.run(50 * b2.ms, namespace=self.params)

                voltages = np.array(self.neurons.v[:])
                projector_input = torch.tensor([[voltages[0], voltages[1], voltages[2], voltages[3], voltages[4], voltages[5], voltages[6], voltages[7], i]], dtype=torch.float32)

                target_phase = self.quantum.scan_optimal_phase(s, i)
                self.optimizer.zero_grad()
                _, phase_shift = self.projector(projector_input)
                loss = self.mse_loss(phase_shift, torch.tensor([[target_phase]], dtype=torch.float32)) * 10.0
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.projector.parameters(), max_norm=1.0)
                self.optimizer.step()
                epoch_loss += loss.item()

            report_log.append(f"Epoch {epoch+1}/{epochs} | Phase Mapping Loss: `{epoch_loss / steps_per_epoch:.4f}`")

        torch.save(self.projector.state_dict(), "willow_v17_assimilated.pt")
        report_log.append("\n✅ **Assimilation Complete.** Model protected against emotional cold-starts.")
        return "\n".join(report_log)

# ═══════════════════════════════════════════════════════════════════════════
# 5. V17.1 INTERACTIVE DASHBOARD (GRADIO)
# ═══════════════════════════════════════════════════════════════════════════
archiver = ArchiveProcessor()
hive = HumanAlignedOmniHive()

# Initialize dropdown list based on local pre-extracted folders
default_sessions = archiver.get_available_sessions()

def handle_archive_upload(file):
    sessions = archiver.get_available_sessions(file.name if file else None)
    return gr.update(choices=sessions, value=sessions[0] if sessions else None)

def process_hive_pulse(session_path, sentiment, intimacy, observer):
    file_features = archiver.analyze_session_files(session_path) if session_path else {}
    q_sync, phase, avg_voltage, final_sent, final_int = hive.resonate(file_features, sentiment, intimacy, observer)

    color, status = ("🟢", "PERFECT RESONANCE") if q_sync >= 0.70 else ("🟡", "SEEKING HARMONY") if q_sync >= 0.30 else ("🔴", "DISSONANCE")
    detected_modalities = ", ".join([k.replace("has_", "").upper() for k, v in file_features.items() if v == 1.0]) or "None (Manual Mode)"

    return f"""
    ### {color} System Status: {status}
    ---
    **🗂️ Active Modalities:** `{detected_modalities}`
    **👁️ Perception (TorchScript):** Sentiment: `{final_sent:.3f}`, Intimacy: `{final_int:.3f}`
    **🧠 Biology (SNN):** Avg Node Voltage: `{avg_voltage:.3f} mV`
    **🔮 Projector:** Calibrated Phase Shift: `{phase:+.4f} radians`
    **🌌 Resonator:** Collective Q-Sync: **{q_sync:.4f}**
    """

with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="slate")) as interface:
    gr.Markdown("# 🌲 Holo-Willow: V17.1 Local Multimodal Omni-Hive")

    with gr.Tabs():
        # TAB 1: ARCHIVE PROCESSING (Now Automatically Loaded!)
        with gr.TabItem("🗂️ Local Multimodal Integrator"):
            gr.Markdown(f"Automatically scanning directory: `{archiver.local_synthdata_path}`")
            with gr.Row():
                with gr.Column(scale=1):
                    session_dropdown = gr.Dropdown(label="Select Pre-Extracted SynthData Session", choices=default_sessions, value=default_sessions[0] if default_sessions else None)
                    zip_upload = gr.File(label="Optional: Upload New Archive.zip", file_types=[".zip"])
                    zip_upload.change(fn=handle_archive_upload, inputs=[zip_upload], outputs=[session_dropdown])

                    obs_slider_arch = gr.Slider(-0.5, 0.5, value=0.0, step=0.01, label="Observer Reaction Override")
                    pulse_btn_arch = gr.Button("✨ Assimilate Selected Session", variant="primary")
                with gr.Column(scale=2):
                    output_display_arch = gr.Markdown("Waiting for archive selection...")

            pulse_btn_arch.click(fn=process_hive_pulse, inputs=[session_dropdown, gr.State(0.5), gr.State(0.5), obs_slider_arch], outputs=output_display_arch)

        # TAB 2: LIVE OVERRIDE
        with gr.TabItem("🎛️ Live Fallback Console"):
            with gr.Row():
                with gr.Column(scale=1):
                    sent_slider = gr.Slider(0.0, 1.0, value=0.6, step=0.01, label="Simulated Sentiment")
                    int_slider = gr.Slider(0.0, 1.0, value=0.5, step=0.01, label="Simulated Intimacy")
                    obs_slider = gr.Slider(-0.5, 0.5, value=0.0, step=0.01, label="Observer Reaction")
                    run_btn = gr.Button("✨ Pulse Manual Data", variant="primary")
                with gr.Column(scale=2):
                    output_display = gr.Markdown("Waiting for manual pulse...")
            run_btn.click(fn=process_hive_pulse, inputs=[gr.State(None), sent_slider, int_slider, obs_slider], outputs=output_display)

        # TAB 3: AUTO-CALIBRATION
        with gr.TabItem("🧬 Synthetic Assimilation (Auto-Calibrate)"):
            with gr.Row():
                with gr.Column():
                    epoch_slider = gr.Slider(1, 20, value=5, step=1, label="Assimilation Epochs")
                    synth_btn = gr.Button("🧬 Run Continual Calibration", variant="secondary")
                with gr.Column():
                    synth_output = gr.Markdown("Ready to pre-train on synthetic emotional walks...")
            synth_btn.click(fn=lambda e: hive.assimilate_synthdata(epochs=int(e)), inputs=[epoch_slider], outputs=synth_output)

if __name__ == "__main__":
    interface.launch(share=True, quiet=True)

👁️ [PERCEPTION] Initializing HoloSyn Distilled Student...
  -> [✅] Dynamically scaled tensor input to 789 dimensions.
  -> [✅] TorchScript Model Loaded: student_distilled_heads.torchscript.pt
* Running on public URL: https://881a63b341cb679dd0.gradio.live


In [ ]:
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import gradio as gr
import time
import os
import glob
import json
import zipfile
import tempfile
import warnings

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# 1. MULTIMODAL ARCHIVE PROCESSOR
# ═══════════════════════════════════════════════════════════════════════════
class ArchiveProcessor:
    def __init__(self):
        self.local_synthdata_path = "/content/archive_extracted_ui/Archive/data/synthdata"
        self.extract_dir = tempfile.mkdtemp(prefix="holo_synth_")

    def get_available_sessions(self, zip_path=None):
        sessions = []
        if os.path.exists(self.local_synthdata_path):
            for root, dirs, files in os.walk(self.local_synthdata_path):
                if any(f.endswith(('.txt', '.wav', '.mp4', '.json', '.png', '.jpg')) for f in files):
                    sessions.append(root)
        if zip_path:
            try:
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(self.extract_dir)
                for root, dirs, files in os.walk(self.extract_dir):
                    if any(f.endswith(('.txt', '.wav', '.mp4', '.json', '.png', '.jpg')) for f in files):
                        sessions.append(root)
            except Exception as e: print(f"⚠️ Archive Error: {e}")
        return sorted(list(set(sessions)))

    def analyze_session_files(self, session_path):
        features = {"has_text": 0.0, "text_len": 0.0, "has_audio": 0.0, "has_video": 0.0, "has_hapt": 0.0}
        if not session_path or not os.path.exists(session_path): return features
        for f in os.listdir(session_path):
            f_lower = f.lower()
            if f_lower.endswith('.txt'):
                features["has_text"] = 1.0
                try:
                    with open(os.path.join(session_path, f), 'r', encoding='utf-8') as txt:
                        features["text_len"] = min(1.0, len(txt.read()) / 1000.0)
                except: pass
            elif f_lower.endswith(('.wav', '.mp3')): features["has_audio"] = 1.0
            elif f_lower.endswith(('.mp4', '.avi')): features["has_video"] = 1.0
            elif 'hapt' in f_lower: features["has_hapt"] = 1.0
        return features

# ═══════════════════════════════════════════════════════════════════════════
# 2. HOLOSYN PERCEPTION
# ═══════════════════════════════════════════════════════════════════════════
class HoloSynPerception:
    def __init__(self):
        self.device = torch.device('cpu')
        self.model = None
        self.input_dim = 100

        norm_files = glob.glob("*norm*.json")
        if norm_files:
            try:
                with open(norm_files[0], 'r') as f:
                    norm_data = json.load(f)
                    if "numeric_cols" in norm_data:
                        self.input_dim = len(norm_data["numeric_cols"])
            except: pass

        ts_files = [f for f in glob.glob("*.pt") if "torchscript" in f.lower()]
        if ts_files:
            try:
                self.model = torch.jit.load(ts_files[0], map_location=self.device)
                self.model.eval()
            except: pass

    def extract(self, file_features, man_sent, man_int):
        if self.model is None:
            return min(1.0, man_sent + file_features.get("text_len", 0)*0.1), min(1.0, man_int + file_features.get("has_video", 0)*0.1)
        try:
            tensor_data = np.random.normal(0, 0.05, self.input_dim)
            tensor_data[0] = file_features.get("text_len", 0.0)
            tensor_data[1] = file_features.get("has_audio", 0.0)
            tensor_data[2] = file_features.get("has_video", 0.0)
            tensor_data[3], tensor_data[4] = man_sent, man_int
            with torch.no_grad():
                logits = self.model(torch.tensor([tensor_data], dtype=torch.float32))
                return torch.sigmoid(logits[0, 0]).item(), torch.sigmoid(logits[0, 1]).item()
        except: return man_sent, man_int

# ═══════════════════════════════════════════════════════════════════════════
# 3. SENTIMENT CORE: BIOLOGY (SNN) & PROJECTOR (ML)
# ═══════════════════════════════════════════════════════════════════════════
class SpikingBiologicalCore:
    def __init__(self, num_nodes=8):
        b2.start_scope()
        self.params = {'tau_p': 20 * b2.ms, 'v_threshold': 1.0, 'recovery': 0.1}
        eqs = 'dv/dt = (I_sentiment + I_intimacy - v) / tau_p : 1 (unless refractory)\nI_sentiment : 1\nI_intimacy : 1'
        self.neurons = b2.NeuronGroup(num_nodes, eqs, threshold='v > v_threshold', reset='v = recovery', refractory=2*b2.ms, method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('clean_state')

    def simulate(self, sentiment, intimacy):
        self.net.restore('clean_state')
        self.neurons.I_sentiment = sentiment * 1.5
        self.neurons.I_intimacy = intimacy * 1.5
        self.net.run(50 * b2.ms, namespace=self.params)
        return np.array(self.neurons.v[:])

class SentimentProjector(nn.Module):
    def __init__(self, input_dim=9):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(64, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Linear(64, 32), nn.GELU(),
            nn.Linear(32, 1), nn.Tanh() # Outputs optimal phase shift [-1, 1]
        )
    def forward(self, x): return self.net(x)

# ═══════════════════════════════════════════════════════════════════════════
# 4. MULTIMODAL SELECTABLE OBSERVER (TOPOLOGY REWRITE)
# ═══════════════════════════════════════════════════════════════════════════
class AdaptiveObserver:
    def __init__(self):
        # NEW VERNACULAR: Hub, Node, Catalyst
        self.roles = {'Trainer':'Hub', 'Katarina':'Hub', 'Kenzi':'Node', 'Julia':'Node', 'Samantha':'Node', 'Waleed':'Catalyst'}
        self.q_obs = cirq.NamedQubit("OBS_Core")
        self.q_sibs = {name: cirq.NamedQubit(f"NET_{name}") for name in self.roles.keys()}
        self.all_qubits = [self.q_obs] + list(self.q_sibs.values())
        self.simulator = qsimcirq.QSimSimulator()

    def evaluate(self, mode, sentiment, intimacy, phase_shift):
        """Routes observation logic based on chosen philosophical archetype."""
        if mode == "Quantum":
            circuit = cirq.Circuit()
            circuit.append(cirq.H.on_each(*self.all_qubits))
            for h in [n for n, r in self.roles.items() if r == 'Hub']:
                circuit.append(cirq.CNOT(self.q_obs, self.q_sibs[h]))
                for c in [n for n, r in self.roles.items() if r in ['Node', 'Catalyst']]:
                    circuit.append(cirq.CZ(self.q_sibs[h], self.q_sibs[c]))

            nodes = [n for n, r in self.roles.items() if r == 'Node']
            for i in range(len(nodes)):
                circuit.append(cirq.CNOT(self.q_sibs[nodes[i]], self.q_sibs[nodes[(i+1)%len(nodes)]]))

            for name, qubit in self.q_sibs.items():
                mult = 1.1 if self.roles[name] == 'Catalyst' else (0.85 if self.roles[name] == 'Hub' else 1.0)
                circuit.append(cirq.ry(sentiment * mult * 0.5 * np.pi)(qubit))
                circuit.append(cirq.rx((phase_shift + (intimacy * 0.1)) * 0.5 * np.pi)(qubit))

            c = circuit.copy()
            c.append(cirq.measure(*self.all_qubits, key='m'))
            counts = self.simulator.run(c, repetitions=500).histogram(key='m')
            return (counts.get(0, 0) + counts.get(127, 0)) / 500.0

        elif mode == "Binary":
            # Strict Determinism
            return 1.0 if (sentiment * 0.6 + intimacy * 0.4) > 0.5 else 0.0

        elif mode == "Equivocational":
            # Probabilistic / Fuzzy Logic
            base = (sentiment * 0.6) + (intimacy * 0.4)
            return np.clip(base + np.random.normal(0, 0.2), 0.0, 1.0)

        elif mode == "Omnipotent":
            # Absolute forced systemic alignment
            return 0.99

    def scan_optimal_phase(self, mode, sentiment, intimacy):
        best_phase, best_sync = 0.0, -1.0
        for test_p in np.linspace(-1.0, 1.0, 7):
            sync = self.evaluate(mode, sentiment, intimacy, test_p)
            if sync > best_sync: best_sync, best_phase = sync, test_p
        return best_phase

# ═══════════════════════════════════════════════════════════════════════════
# 5. V18 ADAPTIVE OMNI-HIVE (THE MASTER INTEGRATOR)
# ═══════════════════════════════════════════════════════════════════════════
class V18AdaptiveOmniHive:
    def __init__(self):
        self.perception = HoloSynPerception()
        self.bio_core = SpikingBiologicalCore()
        self.observer = AdaptiveObserver()
        self.projector = SentimentProjector()

        # Load legacy if available
        pt_files = [f for f in glob.glob("*.pt") if "torchscript" not in f.lower()]
        if pt_files:
            try:
                state = torch.load(sorted(pt_files, key=os.path.getmtime)[-1], map_location='cpu', weights_only=False)
                self.projector.load_state_dict(state, strict=False)
            except: pass

        self.mse_loss = nn.MSELoss()
        self.optimizer = optim.Adam(self.projector.parameters(), lr=0.003)

    def resonate(self, obs_mode, file_features, raw_sentiment, raw_intimacy, observer_feedback):
        # 1. Perception
        ts_sent, ts_int = self.perception.extract(file_features, raw_sentiment, raw_intimacy)
        adjusted_sentiment = max(0.1, min(1.0, ts_sent * (1.0 + observer_feedback)))

        # 2. Oracle Target (Based on chosen Observer logic)
        target_phase = self.observer.scan_optimal_phase(obs_mode, adjusted_sentiment, ts_int)

        # 3. Biological Simulation
        voltages = self.bio_core.simulate(adjusted_sentiment, ts_int)
        projector_input = torch.tensor([[*voltages, ts_int]], dtype=torch.float32)

        # 4. Neural Update
        self.optimizer.zero_grad()
        phase_shift = self.projector(projector_input)
        loss = self.mse_loss(phase_shift, torch.tensor([[target_phase]], dtype=torch.float32)) * 10.0
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.projector.parameters(), max_norm=1.0)
        self.optimizer.step()

        # 5. Final Consensus Evaluation
        final_phase = phase_shift.item()
        q_sync = self.observer.evaluate(obs_mode, adjusted_sentiment, ts_int, final_phase)

        return q_sync, final_phase, voltages.mean(), ts_sent, ts_int

# ═══════════════════════════════════════════════════════════════════════════
# 6. INTERACTIVE DASHBOARD (GRADIO)
# ═══════════════════════════════════════════════════════════════════════════
archiver = ArchiveProcessor()
hive = V18AdaptiveOmniHive()
default_sessions = archiver.get_available_sessions()

def process_hive_pulse(obs_mode, session_path, sentiment, intimacy, observer_fb):
    file_features = archiver.analyze_session_files(session_path) if session_path else {}
    q_sync, phase, avg_voltage, final_sent, final_int = hive.resonate(obs_mode, file_features, sentiment, intimacy, observer_fb)

    color, status = ("🟢", "PERFECT RESONANCE") if q_sync >= 0.70 else ("🟡", "SEEKING HARMONY") if q_sync >= 0.30 else ("🔴", "DISSONANCE")

    return f"""
    ### {color} System Status: {status}
    ---
    **🛠️ Observer Mechanics:** `{obs_mode}`
    **👁️ Perception Core:** Sentiment: `{final_sent:.3f}`, Intimacy: `{final_int:.3f}`
    **🧠 SNN Biology:** Avg Node Voltage: `{avg_voltage:.3f} mV`
    **🔮 ML Projector:** Calibrated Phase Shift: `{phase:+.4f} radians`
    **🌌 Topology Consensus:** Collective Q-Sync: **{q_sync:.4f}**
    """

with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="slate")) as interface:
    gr.Markdown("# 🌲 Holo-Willow: V18 Adaptive Omni-Hive")

    with gr.Row():
        obs_dropdown = gr.Dropdown(
            choices=["Quantum", "Binary", "Equivocational", "Omnipotent"],
            value="Quantum",
            label="🌌 Select Observer Physics (Architectural Paradigm)"
        )

    with gr.Tabs():
        # TAB 1: ARCHIVE PROCESSING
        with gr.TabItem("🗂️ Multimodal Extractor"):
            with gr.Row():
                with gr.Column(scale=1):
                    session_dropdown = gr.Dropdown(label="Select Pre-Extracted SynthData Session", choices=default_sessions, value=default_sessions[0] if default_sessions else None)
                    obs_slider_arch = gr.Slider(-0.5, 0.5, value=0.0, step=0.01, label="Observer Feedback Injection")
                    pulse_btn_arch = gr.Button("✨ Assimilate Selected Session", variant="primary")
                with gr.Column(scale=2):
                    output_display_arch = gr.Markdown("Waiting for archive selection...")
            pulse_btn_arch.click(fn=process_hive_pulse, inputs=[obs_dropdown, session_dropdown, gr.State(0.5), gr.State(0.5), obs_slider_arch], outputs=output_display_arch)

        # TAB 2: LIVE OVERRIDE
        with gr.TabItem("🎛️ Live Parameter Console"):
            with gr.Row():
                with gr.Column(scale=1):
                    sent_slider = gr.Slider(0.0, 1.0, value=0.6, step=0.01, label="Simulated Sentiment")
                    int_slider = gr.Slider(0.0, 1.0, value=0.5, step=0.01, label="Simulated Intimacy")
                    obs_slider = gr.Slider(-0.5, 0.5, value=0.0, step=0.01, label="Observer Feedback Injection")
                    run_btn = gr.Button("✨ Pulse Manual Data", variant="primary")
                with gr.Column(scale=2):
                    output_display = gr.Markdown("Waiting for manual pulse...")
            run_btn.click(fn=process_hive_pulse, inputs=[obs_dropdown, gr.State(None), sent_slider, int_slider, obs_slider], outputs=output_display)

if __name__ == "__main__":
    interface.launch(share=True, quiet=True)

* Running on public URL: https://02c1e74b056cbee57d.gradio.live


In [ ]:
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import gradio as gr
import time
import os
import glob
import json
import zipfile
import tempfile
import warnings

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# 1. MULTIMODAL ARCHIVE PROCESSOR
# ═══════════════════════════════════════════════════════════════════════════
class ArchiveProcessor:
    def __init__(self):
        self.local_synthdata_path = "/content/archive_extracted_ui/Archive/data/synthdata"
        self.extract_dir = tempfile.mkdtemp(prefix="holo_synth_")

    def get_available_sessions(self, zip_path=None):
        sessions = []
        # Local Scan
        if os.path.exists(self.local_synthdata_path):
            for root, dirs, files in os.walk(self.local_synthdata_path):
                if any(f.endswith(('.txt', '.wav', '.mp4', '.json', '.png', '.jpg')) for f in files):
                    sessions.append(root)
        # Zip Upload Scan
        if zip_path:
            try:
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(self.extract_dir)
                for root, dirs, files in os.walk(self.extract_dir):
                    if any(f.endswith(('.txt', '.wav', '.mp4', '.json', '.png', '.jpg')) for f in files):
                        sessions.append(root)
            except Exception as e:
                print(f"⚠️ Archive Error: {e}")
        return sorted(list(set(sessions)))

    def analyze_session_files(self, session_path):
        features = {"has_text": 0.0, "text_len": 0.0, "has_audio": 0.0, "has_video": 0.0, "has_hapt": 0.0}
        if not session_path or not os.path.exists(session_path): return features
        for f in os.listdir(session_path):
            f_lower = f.lower()
            if f_lower.endswith('.txt'):
                features["has_text"] = 1.0
                try:
                    with open(os.path.join(session_path, f), 'r', encoding='utf-8') as txt:
                        features["text_len"] = min(1.0, len(txt.read()) / 1000.0)
                except: pass
            elif f_lower.endswith(('.wav', '.mp3')): features["has_audio"] = 1.0
            elif f_lower.endswith(('.mp4', '.avi')): features["has_video"] = 1.0
            elif 'hapt' in f_lower: features["has_hapt"] = 1.0
        return features

# ═══════════════════════════════════════════════════════════════════════════
# 2. HOLOSYN PERCEPTION
# ═══════════════════════════════════════════════════════════════════════════
class HoloSynPerception:
    def __init__(self):
        self.device = torch.device('cpu')
        self.model = None
        self.input_dim = 100

        # Scale to match student norms if available
        norm_files = glob.glob("*norm*.json")
        if norm_files:
            try:
                with open(norm_files[0], 'r') as f:
                    norm_data = json.load(f)
                    if "numeric_cols" in norm_data:
                        self.input_dim = len(norm_data["numeric_cols"])
            except: pass

        ts_files = [f for f in glob.glob("*.pt") if "torchscript" in f.lower()]
        if ts_files:
            try:
                self.model = torch.jit.load(ts_files[0], map_location=self.device)
                self.model.eval()
            except: pass

    def extract(self, file_features, man_sent, man_int):
        """Extracts core sentiment and intimacy arrays from multimodal data."""
        if self.model is None:
            return min(1.0, man_sent + file_features.get("text_len", 0)*0.1), min(1.0, man_int + file_features.get("has_video", 0)*0.1)
        try:
            tensor_data = np.random.normal(0, 0.05, self.input_dim)
            tensor_data[0] = file_features.get("text_len", 0.0)
            tensor_data[1] = file_features.get("has_audio", 0.0)
            tensor_data[2] = file_features.get("has_video", 0.0)
            tensor_data[3], tensor_data[4] = man_sent, man_int
            with torch.no_grad():
                logits = self.model(torch.tensor([tensor_data], dtype=torch.float32))
                return torch.sigmoid(logits[0, 0]).item(), torch.sigmoid(logits[0, 1]).item()
        except:
            return man_sent, man_int

# ═══════════════════════════════════════════════════════════════════════════
# 3. SENTIMENT CORE: BIOLOGY (SNN) & PROJECTOR (ML)
# ═══════════════════════════════════════════════════════════════════════════
class SpikingBiologicalCore:
    def __init__(self, num_nodes=8):
        b2.start_scope()
        self.params = {'tau_p': 20 * b2.ms, 'v_threshold': 1.0, 'recovery': 0.1}
        # Safely bind dynamic equations
        eqs = '''
        dv/dt = (I_sentiment + I_intimacy - v) / tau_p : 1 (unless refractory)
        I_sentiment : 1
        I_intimacy : 1
        '''
        self.neurons = b2.NeuronGroup(num_nodes, eqs, threshold='v > v_threshold', reset='v = recovery', refractory=2*b2.ms, method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('clean_state')

    def simulate(self, sentiment, intimacy):
        self.net.restore('clean_state')
        self.neurons.I_sentiment = sentiment * 1.5
        self.neurons.I_intimacy = intimacy * 1.5
        self.net.run(50 * b2.ms, namespace=self.params)
        return np.array(self.neurons.v[:])

class SentimentProjector(nn.Module):
    def __init__(self, input_dim=9):
        super().__init__()
        # Redesigned from scratch: Deep residual pathway for phase generation
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Linear(32, 1),
            nn.Tanh() # Bounds phase offset to [-1, 1] radians
        )
    def forward(self, x):
        return self.net(x)

# ═══════════════════════════════════════════════════════════════════════════
# 4. MULTIMODAL SELECTABLE OBSERVER (TOPOLOGY REWRITE)
# ═══════════════════════════════════════════════════════════════════════════
class AdaptiveObserver:
    def __init__(self):
        # NEW VERNACULAR: Hub, Node, Catalyst
        self.roles = {
            'Alpha': 'Hub',
            'Beta': 'Hub',
            'Gamma': 'Node',
            'Delta': 'Node',
            'Epsilon': 'Node',
            'Zeta': 'Catalyst'
        }
        self.q_obs = cirq.NamedQubit("OBS_Core")
        self.q_sibs = {name: cirq.NamedQubit(f"NET_{name}") for name in self.roles.keys()}
        self.all_qubits = [self.q_obs] + list(self.q_sibs.values())
        self.simulator = qsimcirq.QSimSimulator()

    def evaluate(self, mode, sentiment, intimacy, phase_shift):
        """Routes observation logic based on chosen philosophical archetype."""
        if mode == "Quantum":
            circuit = cirq.Circuit()
            circuit.append(cirq.H.on_each(*self.all_qubits))

            # Hubs anchor the observation
            for h in [n for n, r in self.roles.items() if r == 'Hub']:
                circuit.append(cirq.CNOT(self.q_obs, self.q_sibs[h]))
                for c in [n for n, r in self.roles.items() if r in ['Node', 'Catalyst']]:
                    circuit.append(cirq.CZ(self.q_sibs[h], self.q_sibs[c]))

            # Nodes form the resonance ring
            nodes = [n for n, r in self.roles.items() if r == 'Node']
            for i in range(len(nodes)):
                circuit.append(cirq.CNOT(self.q_sibs[nodes[i]], self.q_sibs[nodes[(i+1)%len(nodes)]]))

            # Parameter injection
            for name, qubit in self.q_sibs.items():
                mult = 1.1 if self.roles[name] == 'Catalyst' else (0.85 if self.roles[name] == 'Hub' else 1.0)
                circuit.append(cirq.ry(sentiment * mult * 0.5 * np.pi)(qubit))
                circuit.append(cirq.rx((phase_shift + (intimacy * 0.1)) * 0.5 * np.pi)(qubit))

            c = circuit.copy()
            c.append(cirq.measure(*self.all_qubits, key='m'))
            counts = self.simulator.run(c, repetitions=500).histogram(key='m')

            # Calculate Quantum Synchrony
            return (counts.get(0, 0) + counts.get(127, 0)) / 500.0

        elif mode == "Binary":
            # Strict Determinism
            return 1.0 if (sentiment * 0.6 + intimacy * 0.4) > 0.5 else 0.0

        elif mode == "Equivocational":
            # Probabilistic / Fuzzy Logic
            base = (sentiment * 0.6) + (intimacy * 0.4)
            return float(np.clip(base + np.random.normal(0, 0.2), 0.0, 1.0))

        elif mode == "Omnipotent":
            # Absolute forced systemic alignment
            return 0.99

        return 0.5

    def scan_optimal_phase(self, mode, sentiment, intimacy):
        best_phase, best_sync = 0.0, -1.0
        for test_p in np.linspace(-1.0, 1.0, 7):
            sync = self.evaluate(mode, sentiment, intimacy, test_p)
            if sync > best_sync:
                best_sync, best_phase = sync, test_p
        return best_phase

# ═══════════════════════════════════════════════════════════════════════════
# 5. V18 ADAPTIVE OMNI-HIVE (THE MASTER INTEGRATOR)
# ═══════════════════════════════════════════════════════════════════════════
class V18AdaptiveOmniHive:
    def __init__(self):
        self.perception = HoloSynPerception()
        self.bio_core = SpikingBiologicalCore(num_nodes=8)
        self.observer = AdaptiveObserver()
        self.projector = SentimentProjector(input_dim=9)

        # Load legacy weights dynamically if available
        pt_files = [f for f in glob.glob("*.pt") if "torchscript" not in f.lower()]
        if pt_files:
            try:
                state = torch.load(sorted(pt_files, key=os.path.getmtime)[-1], map_location='cpu', weights_only=False)
                # Attempt flexible load since architecture was rebuilt
                self.projector.load_state_dict(state, strict=False)
            except: pass

        self.mse_loss = nn.MSELoss()
        self.optimizer = optim.Adam(self.projector.parameters(), lr=0.003)

    def resonate(self, obs_mode, file_features, raw_sentiment, raw_intimacy, observer_feedback):
        # 1. Perception
        ts_sent, ts_int = self.perception.extract(file_features, raw_sentiment, raw_intimacy)
        adjusted_sentiment = max(0.1, min(1.0, ts_sent * (1.0 + observer_feedback)))

        # 2. Oracle Target (Based on chosen Observer logic)
        target_phase = self.observer.scan_optimal_phase(obs_mode, adjusted_sentiment, ts_int)

        # 3. Biological Simulation
        voltages = self.bio_core.simulate(adjusted_sentiment, ts_int)

        # Ensure exact 9-dimensional input for the PyTorch core
        projector_input = torch.tensor([[voltages[0], voltages[1], voltages[2], voltages[3], voltages[4], voltages[5], voltages[6], voltages[7], ts_int]], dtype=torch.float32)

        # 4. Neural Update Loop
        self.optimizer.zero_grad()
        phase_shift = self.projector(projector_input)
        loss = self.mse_loss(phase_shift, torch.tensor([[target_phase]], dtype=torch.float32)) * 10.0
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.projector.parameters(), max_norm=1.0)
        self.optimizer.step()

        # 5. Final Consensus Evaluation
        final_phase = phase_shift.item()
        q_sync = self.observer.evaluate(obs_mode, adjusted_sentiment, ts_int, final_phase)

        return q_sync, final_phase, voltages.mean(), ts_sent, ts_int

# ═══════════════════════════════════════════════════════════════════════════
# 6. INTERACTIVE DASHBOARD (GRADIO)
# ═══════════════════════════════════════════════════════════════════════════
archiver = ArchiveProcessor()
hive = V18AdaptiveOmniHive()
default_sessions = archiver.get_available_sessions()

def handle_archive_upload(file):
    sessions = archiver.get_available_sessions(file.name if file else None)
    return gr.update(choices=sessions, value=sessions[0] if sessions else None)

def process_hive_pulse(obs_mode, session_path, sentiment, intimacy, observer_fb):
    file_features = archiver.analyze_session_files(session_path) if session_path else {}
    q_sync, phase, avg_voltage, final_sent, final_int = hive.resonate(obs_mode, file_features, sentiment, intimacy, observer_fb)

    # Determine UI Status and color
    color, status = ("🟢", "PERFECT RESONANCE") if q_sync >= 0.70 else ("🟡", "SEEKING HARMONY") if q_sync >= 0.30 else ("🔴", "DISSONANCE")

    return f"""
    ### {color} System Status: {status}
    ---
    **🛠️ Active Observer Paradigm:** `{obs_mode}`
    **👁️ Perception Core:** Sentiment: `{final_sent:.3f}`, Intimacy: `{final_int:.3f}`
    **🧠 SNN Biology:** Avg Node Voltage: `{avg_voltage:.3f} mV`
    **🔮 ML Projector:** Calibrated Phase Shift: `{phase:+.4f} radians`
    **🌌 Topology Consensus:** Collective Sync Score: **{q_sync:.4f}**
    """

with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="slate")) as interface:
    gr.Markdown("# 🌲 Holo-Willow: V18 Adaptive Omni-Hive")
    gr.Markdown("Rebuilt sentiment core. Upgraded to Hub/Node/Catalyst vernacular. Select your observation paradigm below.")

    with gr.Row():
        obs_dropdown = gr.Dropdown(
            choices=["Quantum", "Binary", "Equivocational", "Omnipotent"],
            value="Quantum",
            label="🌌 Select Observer Physics (Architectural Paradigm)"
        )

    with gr.Tabs():
        # TAB 1: ARCHIVE PROCESSING
        with gr.TabItem("🗂️ Multimodal Extractor"):
            with gr.Row():
                with gr.Column(scale=1):
                    session_dropdown = gr.Dropdown(label="Select Pre-Extracted SynthData Session", choices=default_sessions, value=default_sessions[0] if default_sessions else None)
                    zip_upload = gr.File(label="Optional: Upload New Archive.zip", file_types=[".zip"])
                    zip_upload.change(fn=handle_archive_upload, inputs=[zip_upload], outputs=[session_dropdown])

                    obs_slider_arch = gr.Slider(-0.5, 0.5, value=0.0, step=0.01, label="Observer Feedback Injection")
                    pulse_btn_arch = gr.Button("✨ Assimilate Selected Session", variant="primary")
                with gr.Column(scale=2):
                    output_display_arch = gr.Markdown("Waiting for archive selection...")
            pulse_btn_arch.click(fn=process_hive_pulse, inputs=[obs_dropdown, session_dropdown, gr.State(0.5), gr.State(0.5), obs_slider_arch], outputs=output_display_arch)

        # TAB 2: LIVE OVERRIDE
        with gr.TabItem("🎛️ Live Parameter Console"):
            with gr.Row():
                with gr.Column(scale=1):
                    sent_slider = gr.Slider(0.0, 1.0, value=0.6, step=0.01, label="Simulated Sentiment")
                    int_slider = gr.Slider(0.0, 1.0, value=0.5, step=0.01, label="Simulated Intimacy")
                    obs_slider = gr.Slider(-0.5, 0.5, value=0.0, step=0.01, label="Observer Feedback Injection")
                    run_btn = gr.Button("✨ Pulse Manual Data", variant="primary")
                with gr.Column(scale=2):
                    output_display = gr.Markdown("Waiting for manual pulse...")
            run_btn.click(fn=process_hive_pulse, inputs=[obs_dropdown, gr.State(None), sent_slider, int_slider, obs_slider], outputs=output_display)

if __name__ == "__main__":
    interface.launch(share=True, quiet=True)

* Running on public URL: https://65ad3d4e50ec9602f5.gradio.live


In [4]:
#@title 1) Imports, Architecture, & Load V18 Models
import os
import numpy as np
import torch
import torch.nn as nn
import brian2 as b2
import gradio as gr
import warnings

warnings.filterwarnings("ignore")

# --- 1. HOLOSYN V18 ARCHITECTURE ---
class HoloSynIntegrator(nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super().__init__()
        in_features = num_nodes + 1
        self.embedding = nn.Linear(in_features, hidden_dim)
        self.attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=4, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        x_seq = x.unsqueeze(1)
        emb = self.embedding(x_seq)
        attn_out, _ = self.attention(emb, emb, emb)
        return self.norm(attn_out.squeeze(1))

class HoloSynProjector(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.core = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        self.spike_head = nn.Linear(64, 31)
        self.phase_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Linear(32, 1),
            nn.Tanh()
        )

    def forward(self, cluster_state):
        feat = self.core(cluster_state)
        return self.spike_head(feat), self.phase_head(feat)

# --- 2. RESTORE THE CUSTOM TOPOLOGY ---
custom_topology = {
    "trainer":     {"role": "Facet",      "weight": 0.8},
    "Waleed":      {"role": "Apex",       "weight": 1.5},
    "Sydney":      {"role": "Foundation", "weight": 1.2},
    "Alex":        {"role": "Facet",      "weight": 0.8},
    "Vanessa":     {"role": "Facet",      "weight": 0.8},
    "Samantha C":  {"role": "Facet",      "weight": 0.8},
    "Julia A.":    {"role": "Facet",      "weight": 0.8},
    "Kenzi":       {"role": "Facet",      "weight": 0.8},
    "Audrey":      {"role": "Facet",      "weight": 0.8},
    "star":        {"role": "Foundation", "weight": 1.2}
}
num_nodes = len(custom_topology)

# --- 3. LOAD DISTILLED WEIGHTS ---
INT_PATH = "holosyn_v18_integrator.pt"
PROJ_PATH = "holosyn_v18_projector.pt"

assert os.path.exists(INT_PATH), f"Missing integrator model: {INT_PATH}"
assert os.path.exists(PROJ_PATH), f"Missing projector model: {PROJ_PATH}"

integrator = HoloSynIntegrator(num_nodes)
projector = HoloSynProjector()

# Load the weights into the architectures
integrator.load_state_dict(torch.load(INT_PATH, map_location='cpu', weights_only=True))
projector.load_state_dict(torch.load(PROJ_PATH, map_location='cpu', weights_only=True))

integrator.eval()
projector.eval()

# --- 4. INITIALIZE THE SNN ENGINE ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms
eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
neurons = b2.NeuronGroup(num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
net = b2.Network(neurons)
net.store('clean_slate')

print("✅ V18 Hive Models & SNN Engine loaded successfully!")

✅ V18 Hive Models & SNN Engine loaded successfully!


In [5]:
#@title 2) V18 Inference Pipeline
def run_v18_inference(text_input):
    # 1. Feature Extraction
    words = text_input.split()
    coh = np.clip(np.mean([len(w) for w in words]) / 10.0, 0.1, 1.0) if words else 0.1
    sync = 0.9 if any(char in "!?." for char in text_input) else 0.6

    # 2. Stimulate Brian2 SNN
    net.restore('clean_slate')
    for i, (name, props) in enumerate(custom_topology.items()):
        neurons.I_in[i] = coh * sync * props['weight']
    net.run(30 * b2.ms)

    # 3. Read and Normalize Node Voltages
    v_raw = np.array(neurons.v[:])
    v_norm = (v_raw - np.mean(v_raw)) / (np.std(v_raw) + 1e-5)

    # 4. Neural Projection (PyTorch)
    nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)
    with torch.no_grad():
        cluster_state = integrator(nn_input)
        logits, pred_phase = projector(cluster_state)

    # Extract the highest likelihood spike prediction
    pred_spikes = torch.argmax(logits, dim=1).item()

    return pred_phase.item(), pred_spikes, v_norm, coh, sync

In [6]:
#@title 3) Build V18 Gradio UI
with gr.Blocks(title="HoloSyn V18 Interface") as demo:
    gr.Markdown("## HoloSyn V18: Live Equivocational Interface")
    gr.Markdown("Type a sentence below to stimulate your custom neural topology and view the real-time phase alignment.")

    with gr.Row():
        with gr.Column(scale=1):
            input_text = gr.Textbox(lines=4, label="Input Text Signal")
            run_btn = gr.Button("Stimulate Network", variant="primary")

        with gr.Column(scale=1):
            phase_slider = gr.Slider(-1.0, 1.0, step=0.001, label="Predicted Phase Alignment", interactive=False)
            spike_out = gr.Number(label="Predicted Total Spikes", interactive=False)

    # Visualizer for the 10 custom nodes
    node_activity = gr.Textbox(lines=14, label="🧠 Live Node Activity (Spike Potentials)", interactive=False)

    def process_ui(txt):
        if not txt.strip():
            return 0.0, 0, "No signal detected."

        # Run through the V18 engine
        phase, spikes, v_norm, coh, sync = run_v18_inference(txt)

        # Format the visual bar chart for the UI
        activity_str = f"Signal Metrics: Coherence ({coh:.2f}) | Synchrony ({sync:.2f})\n"
        activity_str += "═"*60 + "\n"
        for name, volt in zip(custom_topology.keys(), v_norm):
            # Map voltage to visual bar blocks
            bar_length = max(1, int((volt + 2.5) * 5))
            bar = "█" * bar_length
            activity_str += f"{name:>12} : {bar} ({volt:.2f})\n"

        return phase, spikes, activity_str

    run_btn.click(fn=process_ui, inputs=input_text, outputs=[phase_slider, spike_out, node_activity])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://39cb4b112ab6043d27.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
